# LoRaS-CT Notebook — Multi-Dataset Edition (R2 + R3 merged)

This merges the cleaned/de-duplicated **R2** notebook with the **R3** standalone fix for Section 18 (the Hugging Face / CONCH / Virchow foundation-model comparison), and adds support for running the entire pipeline on **five datasets**:

- Kather5k (original)
- CRC7k
- NCT100k
- LC25000
- BreakHis

## How this notebook is organized now

1. **Setup** (installs, imports, seeds, global knobs) — run once.
2. **Dataset Registry & Selector** — edit `DATASET_CONFIGS` with your real dataset paths, then set `SELECTED_DATASET` to one dataset name, or `"ALL"` to run every dataset back-to-back.
3. **Canonical class/function definitions** (CNN feature extractors, LoRaS-CT, MHA-Net, attention baselines, training/eval utilities, GradCAM, foundation-model loaders, etc.) — defined once, dataset-independent.
4. **`run_pipeline(CURRENT_DATASET, DATASET_PATH)`** — ONE function containing the entire per-dataset pipeline (data loading → training → every table/plot/explainability section). Everything it prints, tables it builds, and images it saves are labeled with the dataset that produced them (images go to `images/<dataset_name>/...`).
5. **Driver cell** at the very end — loops over `DATASETS_TO_RUN` and calls `run_pipeline(...)` for each one, collecting results into `all_dataset_results[dataset_name]`.

**Trade-off:** because `run_pipeline` is one big function, you can no longer step through Sections 3–22 cell-by-cell for debugging one dataset at a time the way the original notebook allowed — you run the whole pipeline (for one dataset) by calling `run_pipeline(...)` once, or let the driver cell loop it for all selected datasets.


## 1. Setup

In [ ]:
# Installs
# NOTE: this notebook was originally written for Kaggle, whose default notebook image
# comes with numpy/pandas/matplotlib/seaborn/torch/torchvision/scikit-learn/scipy/timm
# already preinstalled. Running elsewhere (plain local Jupyter, a fresh venv, Colab,
# etc.) starts with none of that, so we install the full stack explicitly below --
# this is a no-op (just confirms versions) if everything's already present.
#
# GPU note: this installs the CPU/default PyPI build of torch. If you have an NVIDIA
# GPU and want CUDA acceleration, install torch/torchvision FIRST from
# https://pytorch.org/get-started/locally/ (pick your CUDA version there), THEN re-run
# this cell -- pip will see torch is already satisfied and skip reinstalling it.

# Upgrade pip's own build tooling first -- on some local/offline environments pip's
# bundled setuptools is too old (or missing) to build packages that need it, which
# surfaces as "Could not find a version that satisfies the requirement setuptools>=61.0"
# during the *next* install's build-dependency step. This line is a no-op if you're
# already up to date.
!pip install --upgrade pip setuptools wheel --quiet

!pip install numpy pandas matplotlib seaborn scipy sympy --quiet
!pip install torch torchvision --quiet
!pip install timm scikit-learn scikit-image --quiet
!pip install thop shap lime --quiet

import os
os.environ['HF_HUB_DISABLE_XET'] = '1'

In [ ]:
!pip install -q -U huggingface_hub hf_xet

In [ ]:
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'

In [ ]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.functional import softmax
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split
from timm import create_model
from timm.layers import DropPath          # <-- yahi change kiya
from thop import profile
from sklearn.manifold import TSNE
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, auc,
)

random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)

if not os.path.exists("images"):
    os.makedirs("images")

# ============================================================
# QUICK TEST MODE -- smoke-test the ENTIRE notebook end-to-end in ~10-15 min
# before committing to the full multi-hour run. Every section that trains
# multiple models (14, 17, 18, 19, 20, 21) and the main comparison (8) respect
# this flag automatically -- no other cell needs to be touched.
#
#   QUICK_TEST_MODE = True   -> tiny data subsets, fewer seeds/folds/configs,
#                                foundation-model downloads skipped. Verifies
#                                every cell RUNS without crashing. Numbers
#                                will NOT be meaningful (too little data).
#   QUICK_TEST_MODE = False  -> the real run. Use this for camera-ready numbers.
# ============================================================
QUICK_TEST_MODE = False

# ============================================================
# GLOBAL EPOCH CONTROL -- single knob for every training section.
# Every *_EPOCHS constant below (Sections 8, 14, 17, 18, 19, 20, 21)
# now just reads this value, so changing epoch count for the whole
# notebook means editing ONE line here instead of eight.
# ============================================================
GLOBAL_NUM_EPOCHS = 10

# ============================================================
# GLOBAL RANK CONTROL -- single knob for the LoRaS-CT low-rank
# dimension R. Every rank=32 default/call across the notebook
# (main model, Grad-CAM visualizable twin, single-backbone
# ablations, w/o-sparsity ablation, and the two demo/profiling
# cells) now reads this value instead of hardcoding 32.
# ============================================================
GLOBAL_RANK = 32

# ============================================================
# GLOBAL LAYERS/HEADS CONTROL -- single knobs for the transformer
# depth (num_layers) and multi-head split (num_heads), matching the
# GLOBAL_RANK / GLOBAL_NUM_EPOCHS pattern above. Every num_layers=2 /
# num_heads=8 default across the notebook (main models, visualizable
# twin, single-backbone ablation) now reads these instead of hardcoding.
# NOTE: rank must stay divisible by num_heads (LowRankSparseMultiheadAttention
# asserts this) -- if you change GLOBAL_NUM_HEADS, make sure it still divides
# GLOBAL_RANK evenly.
# ============================================================
GLOBAL_NUM_LAYERS = 2
GLOBAL_NUM_HEADS = 8

# Set True to ignore any saved checkpoints and force every model to retrain
# from scratch (e.g. after you've changed hyperparameters).
FORCE_RETRAIN = False

QUICK_TEST_MAX_TRAIN = 64
QUICK_TEST_MAX_VAL = 16
QUICK_TEST_MAX_TEST = 16
QUICK_TEST_BATCH_SIZE = 8

from torch.utils.data import Subset

def quick_subset(torch_dataset, max_n):
    """If QUICK_TEST_MODE, caps a Dataset/Subset to its first max_n samples.
    Deterministic (not random) so repeated smoke-test runs are comparable.
    Safe to call on an already-nested Subset -- Section 22's leakage audit
    walks through arbitrarily many layers of Subset, so this doesn't break it.
    A no-op when QUICK_TEST_MODE is False."""
    if not QUICK_TEST_MODE:
        return torch_dataset
    n = min(max_n, len(torch_dataset))
    return Subset(torch_dataset, list(range(n)))

if QUICK_TEST_MODE:
    print("QUICK_TEST_MODE is ON -- using tiny data subsets to smoke-test the notebook.")
    print("Set QUICK_TEST_MODE = False above before running for real results.\n")

## 2. Dataset Registry & Selector

Add/edit dataset paths below. Every dataset must be laid out **ImageFolder-style** (one subfolder per class), e.g.:

```
/path/to/CRC7k/
    ClassA/xxx.png
    ClassB/yyy.png
    ...
```

The four new datasets below use **placeholder paths** — replace them with your actual locations before running. `num_classes` and class names are auto-detected from each dataset's folder structure at run time, so nothing else needs to change per dataset.

In [ ]:
# ============================================================
# DATASET REGISTRY -- replace the placeholder paths below with your real
# dataset locations. Kather5k keeps the original path from the notebook.
#
# Two supported shapes per entry:
#   1. "path": a single folder containing one sub-folder per class
#      (loaded directly with ImageFolder; this notebook does its own
#      80/10/10-ish train/val/test random_split on top).
#   2. "train_val_path" + "test_path": for datasets that ship with their
#      own official train/val and test split already separated (e.g.
#      LC25000's "Train and Validation Set" / "Test Set" folders). The
#      official test split is kept as-is (not re-split); only the
#      train_val folder gets our own 90/10 train/val split. Both folders
#      must contain identically-named class sub-folders.
# ============================================================
# Detect Kaggle by an env var Kaggle itself sets (KAGGLE_KERNEL_RUN_TYPE), NOT by
# checking whether /kaggle/input or /kaggle/working happen to exist as directories --
# on some servers/containers those paths exist for unrelated reasons (stale mounts,
# leftover dirs, etc.), which was wrongly triggering the Kaggle branch below and
# pointing DATA_ROOT/OUTPUT_ROOT at paths that aren't actually usable here.
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

# DATA_ROOT lets the same config work on Kaggle (where datasets are mounted under
# /kaggle/input) and locally/elsewhere (where you point this at wherever you've put the
# data on disk). Override DATA_ROOT yourself (env var) if your data lives somewhere else.
DATA_ROOT = "/kaggle/input" if IS_KAGGLE else os.environ.get("DATA_ROOT", "./data")

# Same portability fix for OUTPUTS: Kaggle notebooks write checkpoints/results to
# /kaggle/working, which only exists (and is writable) on actual Kaggle. Elsewhere, this
# falls back to a local "./outputs" folder (created automatically) -- override with an
# OUTPUT_ROOT env var if you want checkpoints written somewhere else.
OUTPUT_ROOT = "/kaggle/working" if IS_KAGGLE else os.environ.get("OUTPUT_ROOT", "./outputs")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

DATASET_CONFIGS = {
    "Kather5k": {
        "path": "/scratch/home/admin/Kather_texture_2016_image_tiles_5000",
    },
    "CRC7k": {
        "path": "/scratch/home/admin//CRC-VAL-HE-7K",        # <-- replace with your CRC7k path
    },
    "NCT100k": {
        "path": f"{DATA_ROOT}/PLACEHOLDER_PATH/NCT100k",      # <-- replace with your NCT100k path
    },
    "LC25000": {
        # Pre-split dataset: official Test Set kept separate; only
        # "Train and Validation Set" gets our own 90/10 train/val split.
        "train_val_path": f"{DATA_ROOT}/datasets/javaidahmadwani/lc25000/lung_colon_image_set/Train and Validation Set"
                           if IS_KAGGLE else f"{DATA_ROOT}/LC25000/Train and Validation Set",
        "test_path": f"{DATA_ROOT}/datasets/javaidahmadwani/lc25000/lung_colon_image_set/Test Set"
                     if IS_KAGGLE else f"{DATA_ROOT}/LC25000/Test Set",
    },
    "BreakHis": {
        "path": "/scratch/home/admin/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/",     # <-- replace with your BreakHis path
    },
}

# Pre-flight check: verify each selected dataset's path(s) actually exist BEFORE the
# pipeline runs, with a clear message instead of a bare FileNotFoundError mid-training.
def _check_dataset_paths(cfg):
    paths = [cfg["path"]] if "path" in cfg else [cfg["train_val_path"], cfg["test_path"]]
    return [p for p in paths if not os.path.isdir(p)]

# ============================================================
# DATASET SELECTOR
#   - one name from DATASET_CONFIGS above  -> runs the pipeline for just that dataset
#   - "ALL"                                -> runs the pipeline once per dataset, back-to-back
# ============================================================
SELECTED_DATASET = "CRC7k"   # <-- change this: "Kather5k" | "CRC7k" | "NCT100k" | "LC25000" | "BreakHis" | "ALL"

assert SELECTED_DATASET == "ALL" or SELECTED_DATASET in DATASET_CONFIGS, (
    f"Unknown SELECTED_DATASET '{SELECTED_DATASET}'. "
    f"Choose one of {list(DATASET_CONFIGS)} or 'ALL'."
)
DATASETS_TO_RUN = list(DATASET_CONFIGS.keys()) if SELECTED_DATASET == "ALL" else [SELECTED_DATASET]

print(f"Will run the pipeline for: {DATASETS_TO_RUN}")


In [ ]:
# ============================================================
# Patient/slide-level leakage-fix helpers -- used by the main split (Section 2),
# make_split() (Section 17, multi-seed), and K-Fold (Section 19) so that BreakHis
# (which has multiple patches per patient) never puts the same patient in more
# than one split. Datasets with no parseable patient ID in their filenames
# (Kather5k, NCT100k -- tile-level benchmarks) fall back to the original
# image-level random split automatically -- nothing changes for them.
# ============================================================
import re
from sklearn.model_selection import GroupShuffleSplit

def try_extract_patient_id(filename):
    """Same patterns as the Section 22 audit -- kept in sync with it."""
    patterns = [
        r'SOB_[A-Z]_[A-Z]+-(\d+-\d+)',   # BreakHis: SOB_B_TA-14-4659-40-001.png -> "14-4659"
        r'^(P\d+)',
        r'patient[_-]?(\d+)',
        r'case[_-]?(\d+)',
    ]
    for pat in patterns:
        m = re.search(pat, filename, flags=re.IGNORECASE)
        if m:
            return m.group(1)
    return None


def get_patient_groups(image_folder_dataset):
    """Patient/case ID per sample, aligned with .samples order. Returns None
    (caller falls back to plain random split) if filenames don't carry a
    consistent, parseable patient ID -- e.g. Kather5k/NCT100k tile-level data."""
    filepaths = [s[0] for s in image_folder_dataset.samples]
    patient_ids = [try_extract_patient_id(os.path.basename(fp)) for fp in filepaths]
    if any(p is None for p in patient_ids) or len(set(patient_ids)) < 2:
        return None
    return np.array(patient_ids)


def patient_group_split(image_folder_dataset, seed, test_frac=0.2, val_frac_of_trainval=0.1):
    """Patient-level GroupShuffleSplit -- no patient appears in more than one
    split. Returns None if this dataset has no parseable patient grouping."""
    groups = get_patient_groups(image_folder_dataset)
    if groups is None:
        return None

    idx = np.arange(len(image_folder_dataset))
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(idx, groups=groups))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_frac_of_trainval, random_state=seed)
    tr_sub, val_sub = next(gss2.split(trainval_idx, groups=groups[trainval_idx]))
    train_idx, val_idx = trainval_idx[tr_sub], trainval_idx[val_sub]

    assert not (set(groups[train_idx]) & set(groups[test_idx]))
    assert not (set(groups[train_idx]) & set(groups[val_idx]))
    assert not (set(groups[val_idx]) & set(groups[test_idx]))
    return train_idx, val_idx, test_idx


## 3. Canonical class & function definitions (dataset-independent, defined once)

CNN feature extractors, the LoRaS-CT / MHA-Net / attention-baseline model classes, shared training & eval utilities, GradCAM, ablation-only model variants, and the (fixed) pathology foundation-model loaders. None of these reference a specific dataset -- `num_classes` etc. are passed in as arguments at call time inside `run_pipeline` below.

In [ ]:
# ============================================================
# Shared image transforms + batch size -- dataset-independent, defined once.
# (Used by data loading inside run_pipeline AND by make_split() in Section 14.)
# ============================================================
train_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

batch_size = QUICK_TEST_BATCH_SIZE if QUICK_TEST_MODE else 32


### 3.1 Teacher-model imports note
Teacher ViT/DeiT/Swin models are now loaded **inside** `run_pipeline` (their classification head size depends on `num_classes`, which differs per dataset).

In [ ]:
class ResNet18_Features(nn.Module):
    """Returns spatial feature map (B, 512, 7, 7) for a 224x224 input."""
    def __init__(self):
        super(ResNet18_Features, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-2])  # drop avgpool + fc

    def forward(self, x):
        return self.features(x)  # (B, 512, 7, 7)


class DenseNet121_Features(nn.Module):
    """Returns spatial feature map (B, 1024, 7, 7) for a 224x224 input."""
    def __init__(self):
        super(DenseNet121_Features, self).__init__()
        densenet = models.densenet121(pretrained=True)
        self.features = densenet.features

    def forward(self, x):
        x = self.features(x)
        x = F.relu(x, inplace=False)  # inplace=False -- avoids breaking SHAP DeepExplainer's backward hooks
        return x  # (B, 1024, 7, 7)

In [ ]:
class LowRankSparseMultiheadAttention(nn.Module):
    """Low-Rank Sparse Multi-Head Attention -- attention is now computed
    ENTIRELY IN RANK-SPACE (dimension r); Q, K, V are never projected back
    up to full embed_dim (E) before the attention product, unlike the
    earlier "reconstruct-then-attend" version.

    Complexity (matches the paper's O(N^2 r + N r^2) claim for the
    attention-mechanism cost itself):
      - Q,K,V down-projections (E -> r, shared across heads): O(N E r)
      - attention scores + weighted sum, computed at rank r: O(N^2 r)
      - r x r inter-head mixing (rank_mix): O(N r^2)
      - final up-projection back to E (r -> E): O(N E r)
    The O(N^2 r) and O(N r^2) terms are the two that scale with sequence
    length N and dominate the attention block's own cost; the O(NEr) terms
    are the (unavoidable) cost of moving data in/out of rank space once.

    Key points:
      - q_low/k_low/v_low (E -> r) are SHARED across all heads (one E->r
        matrix per Q/K/V, not one per head).
      - Q, K, V STAY at rank r for the entire attention computation --
        there is no q_high/k_high/v_high projection back to E anymore.
      - Top-k sparsity masking is applied per head, at rank dimension.
      - After the weighted sum, heads are concatenated back to r (not E)
        and passed through a small learned r x r mixing matrix (shared
        across heads) -- this is the O(N r^2) term.
      - A single r -> E output projection restores full dimensionality
        ONCE, at the very end (much smaller than the old E x E out_proj).
    """
    def __init__(self, embed_dim, num_heads, rank, sparsity_ratio=0.5):
        super(LowRankSparseMultiheadAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        assert rank % num_heads == 0, "rank must be divisible by num_heads (rank-space multi-head split)"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.rank = rank
        self.rank_head_dim = rank // num_heads  # per-head dim, now IN RANK SPACE (not E // H)
        self.sparsity_ratio = sparsity_ratio

        # Shared low-rank down-projections only (E -> r). No up-projection to E here --
        # Q, K, V are used directly at rank r for the attention product.
        self.q_low = nn.Linear(embed_dim, rank, bias=False)
        self.k_low = nn.Linear(embed_dim, rank, bias=False)
        self.v_low = nn.Linear(embed_dim, rank, bias=False)

        # Small r x r matrix, shared across heads, mixing the concatenated
        # rank-r attention output before the final up-projection. O(N r^2) term.
        self.rank_mix = nn.Linear(rank, rank, bias=False)

        # Final up-projection back to embed_dim (r -> E), replaces the old E x E out_proj.
        self.out_proj = nn.Linear(rank, embed_dim, bias=False)

        # Scale by sqrt(r) -- the actual dimension attention scores are computed in now,
        # not sqrt(E).
        self.scale = rank ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        """Masks out non-top-k positions with -inf BEFORE softmax (not by
        zeroing the score and softmaxing over everything) -- zeroing a raw
        logit and then softmaxing still gives that position exp(0)=1 in the
        numerator, which can outweigh genuinely-kept positions whenever their
        raw scores are negative (common with near-zero-mean post-LayerNorm
        logits, especially early in training). masked_fill(..., -inf) makes
        softmax assign exactly 0 probability to masked positions, which is
        what top-k sparse attention is supposed to do."""
        batch_size, num_heads, seq_length, _ = attn_scores.size()
        if seq_length == 1:
            return attn_scores
        num_to_keep = max(1, int(sparsity_ratio * seq_length))
        top_scores, _ = torch.topk(attn_scores, k=num_to_keep, dim=-1)
        threshold = top_scores.min(dim=-1, keepdim=True)[0]
        sparse_mask = attn_scores >= threshold
        return attn_scores.masked_fill(~sparse_mask, float('-inf'))

    def forward(self, x):
        batch_size, seq_length, embed_dim = x.size()

        # Down-project to rank r -- Q, K, V NEVER return to full E before attention.
        Q = self.q_low(x)  # (B, T, r)
        K = self.k_low(x)  # (B, T, r)
        V = self.v_low(x)  # (B, T, r)

        # Multi-head split WITHIN rank-space: r -> (H, r/H), not E -> (H, E/H).
        Q = Q.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)  # (B,H,T,r/H)
        K = K.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)

        # Attention scores computed AT RANK DIMENSION -> O(N^2 r), not O(N^2 E).
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale  # (B,H,T,T)

        sparse_attn_scores = self.sparse_attention(attn_scores, self.sparsity_ratio)
        attn_probs = F.softmax(sparse_attn_scores, dim=-1)

        attn_output = torch.matmul(attn_probs, V)  # (B,H,T,r/H)

        # Concatenate heads back to full RANK r (not E).
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.rank)

        # r x r inter-head mixing -> O(N r^2).
        attn_output = self.rank_mix(attn_output)

        # Single final up-projection, r -> E.
        return self.out_proj(attn_output)

class CustomDeiTLayer(nn.Module):
    """Transformer encoder layer used by LoRaS-CT."""
    def __init__(self, embed_dim, num_heads, rank, mlp_ratio=4., drop_path=0.1, sparsity_ratio=0.5):
        super(CustomDeiTLayer, self).__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LowRankSparseMultiheadAttention(embed_dim, num_heads, rank, sparsity_ratio)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class HybridStudentModel(nn.Module):
    """LoRaS-CT student model (spatial tokens; grid_size controls N).
    grid_size=7 -> 49 tokens (native CNN output, no reduction)
    grid_size=3 -> 9 tokens (pooled down via adaptive_avg_pool2d) -- matches the diagram
    """
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 rank=GLOBAL_RANK, drop_path_rate=0.1, sparsity_ratio=0.5, grid_size=3):
        super(HybridStudentModel, self).__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        self.grid_size = grid_size
        concat_channels = 512 + 1024  # 1536

        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                             sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x, return_features=False):
        resnet_feats = self.resnet(x)      # (B, 512, 7, 7)
        densenet_feats = self.densenet(x)  # (B, 1024, 7, 7)
        combined_feats = torch.cat((resnet_feats, densenet_feats), dim=1)  # (B, 1536, 7, 7)

        if self.grid_size != combined_feats.shape[-1]:
            combined_feats = F.adaptive_avg_pool2d(combined_feats, (self.grid_size, self.grid_size))

        b, c, h, w = combined_feats.shape
        combined_feats = combined_feats.view(b, c, h * w).permute(0, 2, 1)  # (B, N, 1536)

        x = self.deit_embed(combined_feats)  # (B, N, embed_dim)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        pooled = x.mean(dim=1)  # aggregate N tokens
        logits = self.classifier(pooled)
        if return_features:
            return logits, pooled
        return logits

In [ ]:
class CustomDeiTLayer_MHA(nn.Module):
    """Baseline layer using standard multi-head attention (full complexity),
    written with explicit nn.Linear ops so thop can profile its FLOPs --
    nn.MultiheadAttention is opaque to thop and silently reports 0 FLOPs,
    which was making MHA-Net's FLOPs look lower than LoRaS-CT's."""
    def __init__(self, embed_dim, num_heads, mlp_ratio=4., drop_path=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5
        self.drop_path = nn.Identity() if drop_path == 0 else nn.Dropout(drop_path)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        normed = self.norm1(x)
        B, N, E = normed.shape
        Q = self.q_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        attn_out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        attn_out = self.out_proj(attn_out)
        x = x + self.drop_path(attn_out)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class MHANetBaseline(nn.Module):
    """Same spatial CNN backbone as HybridStudentModel, but standard MHA (for fair comparison)."""
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 drop_path_rate=0.1, grid_size=3):
        super().__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        concat_channels = 512 + 1024
        self.grid_size = grid_size  # must match HybridStudentModel's grid_size for fair comparison

        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer_MHA(embed_dim, num_heads, drop_path=drop_path_rate)
            for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        resnet_feats = self.resnet(x)
        densenet_feats = self.densenet(x)
        combined_feats = torch.cat((resnet_feats, densenet_feats), dim=1)

        if self.grid_size != combined_feats.shape[-1]:
            combined_feats = F.adaptive_avg_pool2d(combined_feats, (self.grid_size, self.grid_size))

        b, c, h, w = combined_feats.shape
        combined_feats = combined_feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(combined_feats)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        x = self.classifier(x)
        return x

In [ ]:
# ============================================================
# New attention baselines for Table 10 (Reviewer 1, Comment 1)
# Drop-in replacements: forward(x) with x of shape (B, N, embed_dim)
# ============================================================

class LinformerAttention(nn.Module):
    """Projects K, V along the sequence dimension N -> k (Wang et al., 2020)."""
    def __init__(self, embed_dim, num_heads, seq_len, proj_k=None):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.seq_len = seq_len
        self.proj_k = proj_k or max(1, seq_len // 2)  # k < N

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.E_proj = nn.Linear(seq_len, self.proj_k, bias=False)  # K: N -> k
        self.F_proj = nn.Linear(seq_len, self.proj_k, bias=False)  # V: N -> k
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        assert N == self.seq_len, f"LinformerAttention was built for N={self.seq_len}, got {N}"
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)          # B,H,N,hd
        K = self.E_proj(self.k_proj(x).transpose(1, 2)).transpose(1, 2)                        # B,k,E
        V = self.F_proj(self.v_proj(x).transpose(1, 2)).transpose(1, 2)                        # B,k,E
        K = K.view(B, self.proj_k, self.num_heads, self.head_dim).transpose(1, 2)              # B,H,k,hd
        V = V.view(B, self.proj_k, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)            # B,H,N,k
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class PerformerAttention(nn.Module):
    """FAVOR+-style linear attention with a fixed (non-trainable) random feature map
    (Choromanski et al., 2021)."""
    def __init__(self, embed_dim, num_heads, nb_features=None):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.nb_features = nb_features or self.head_dim

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        # fixed random projection -- buffer, not a learned parameter
        self.register_buffer(
            'random_matrix', torch.randn(self.num_heads, self.head_dim, self.nb_features)
        )

    def _phi(self, x):  # positive random feature map
        proj = torch.einsum('bhnd,hdf->bhnf', x, self.random_matrix)
        return F.elu(proj) + 1

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        Qp, Kp = self._phi(Q), self._phi(K)
        KV = torch.einsum('bhnf,bhnd->bhfd', Kp, V)
        denom = torch.einsum('bhnf,bhf->bhn', Qp, Kp.sum(dim=2)) + 1e-6
        out = torch.einsum('bhnf,bhfd->bhnd', Qp, KV) / denom.unsqueeze(-1)
        out = out.transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class BigBirdAttention(nn.Module):
    """Block/local/global/random sparse attention (Zaheer et al., 2020).
    Falls back to full dense attention whenever N <= block_size, matching the
    reference implementation's behavior for short sequences."""
    def __init__(self, embed_dim, num_heads, block_size=64):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.block_size = block_size

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        # N is always << block_size in this architecture -> dense fallback
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class StandardMHAWrapper(nn.Module):
    """Standard multi-head attention, written with explicit nn.Linear ops so
    thop can profile its FLOPs (nn.MultiheadAttention is opaque to thop and
    silently reports 0 FLOPs). Parameter count is identical to
    nn.MultiheadAttention: 3 in-projections + 1 out-projection, each E->E
    with bias, i.e. 4E^2 + 4E."""
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class GenericAttnLayer(nn.Module):
    """Same transformer-layer shell as CustomDeiTLayer, but takes any attn module."""
    def __init__(self, attn_module, embed_dim, mlp_ratio=4., drop_path=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = attn_module
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden), nn.GELU(), nn.Linear(mlp_hidden, embed_dim)
        )

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class GenericHybridModel(nn.Module):
    """Same CNN backbone / embed / pooling / classifier as HybridStudentModel and
    MHANetBaseline, so the comparison isolates the attention mechanism only."""
    def __init__(self, num_classes, attn_factory, embed_dim=768, num_layers=GLOBAL_NUM_LAYERS,
                 grid_size=3, drop_path_rate=0.1):
        super().__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        self.grid_size = grid_size
        concat_channels = 512 + 1024

        self.layers = nn.ModuleList([
            GenericAttnLayer(attn_factory(), embed_dim, drop_path=drop_path_rate)
            for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        r = self.resnet(x)
        d = self.densenet(x)
        feats = torch.cat((r, d), dim=1)
        if self.grid_size != feats.shape[-1]:
            feats = F.adaptive_avg_pool2d(feats, (self.grid_size, self.grid_size))
        b, c, h, w = feats.shape
        feats = feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(feats)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.classifier(x)

In [ ]:
class DistillationLoss(nn.Module):
    def __init__(self, alpha=0.5, temperature=3.0):
        super(DistillationLoss, self).__init__()
        self.alpha = alpha
        self.temperature = temperature
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_div = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, ground_truth):
        hard_loss = self.ce_loss(student_logits, ground_truth)
        soft_loss = self.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=1),
            F.softmax(teacher_logits / self.temperature, dim=1)
        ) * (self.temperature ** 2)
        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss


def evaluate_model(model, data_loader, criterion, return_predictions=False):
    model.eval()
    correct, total, test_loss = 0, 0, 0.0
    all_labels, all_preds = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.cuda(), labels.cuda()
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            if return_predictions:
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
    accuracy = 100 * correct / total
    avg_loss = test_loss / len(data_loader)
    if return_predictions:
        return accuracy, avg_loss, np.array(all_labels), np.array(all_preds)
    return accuracy, avg_loss


def train_model_with_distillation(student_model, teacher_models, train_loader, val_loader,
                                   distillation_criterion, optimizer, num_epochs=1):
    train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []
    for epoch in range(num_epochs):
        student_model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            student_outputs = student_model(images)
            with torch.no_grad():
                teacher_logits = [teacher(images) for teacher in teacher_models]
                combined_teacher_logits = sum(teacher_logits) / len(teacher_logits)
            loss = distillation_criterion(student_outputs, combined_teacher_logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)
        train_accuracy, _ = evaluate_model(student_model, train_loader, distillation_criterion.ce_loss)
        val_accuracy, val_loss = evaluate_model(student_model, val_loader, distillation_criterion.ce_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, '
              f'Train Acc: {train_accuracy:.2f}%, Val Acc: {val_accuracy:.2f}%')
    return student_model, train_losses, val_losses, train_accuracies, val_accuracies


def train_model_plain(model, train_loader, val_loader, criterion, optimizer, num_epochs=20):
    """Plain (non-distillation) training loop, used by the standard-CE-loss variants."""
    train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)
        train_accuracy, _ = evaluate_model(model, train_loader, criterion)
        val_accuracy, val_loss = evaluate_model(model, val_loader, criterion)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        print(f"Epoch [{epoch+1}/{num_epochs}], "
              f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")
    return train_losses, val_losses, train_accuracies, val_accuracies


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_custom_parameters(model, exclude_list=None):
    """Count trainable parameters excluding those from specified submodules
    (e.g. exclude the pretrained ResNet/DenseNet backbones)."""
    exclude_list = exclude_list or []
    excluded_params = set(p for submodule in exclude_list for p in submodule.parameters())
    return sum(p.numel() for p in model.parameters() if p.requires_grad and p not in excluded_params)


def measure_inference_time(model, dummy_input, n_runs=50, device='cuda'):
    model.eval()
    with torch.no_grad():
        for _ in range(5):  # warm-up
            _ = model(dummy_input)
    if device == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(dummy_input)
    if device == 'cuda':
        torch.cuda.synchronize()
    end = time.time()
    return (end - start) / n_runs * 1000  # ms


def measure_memory(model, dummy_input, device='cuda'):
    if device != 'cuda':
        return None
    torch.cuda.reset_peak_memory_stats(device)
    model.eval()
    with torch.no_grad():
        _ = model(dummy_input)
    return torch.cuda.max_memory_allocated(device) / (1024 ** 2)  # MB


def measure_inference_time_and_memory(model, data_loader, device='cuda'):
    """FIXED: the original version called torch.cuda.memory_allocated() BEFORE the
    forward pass (measuring stale/leftover memory from whatever ran previously, not
    this model's own usage) and never reset peak-memory stats or synchronized around
    the timer -- which is why two different models could show an IDENTICAL memory
    figure. This version resets peak stats once, runs the whole loader, synchronizes
    properly, and reads true peak memory via max_memory_allocated(). NOTE: Table 5
    below no longer calls this function -- it now reuses comparison_results directly
    (see Section 8) so there is a single source of truth for these numbers. This fixed
    function is kept as a correct general-purpose utility for ad-hoc use elsewhere."""
    model.eval()
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize()

    start_time = time.time()
    n_images = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            n_images += images.size(0)
            outputs = model(images)
    if device == 'cuda':
        torch.cuda.synchronize()
    end_time = time.time()

    total_time = end_time - start_time
    avg_time_per_image_ms = (total_time / n_images) * 1000
    peak_memory = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device == 'cuda' else None

    return peak_memory, total_time, avg_time_per_image_ms


def measure_efficiency_isolated(models_dict, dummy_input, device='cuda', n_runs=50, warmup=5):
    """Measures Peak Memory + Inference Time for EVERY model in models_dict, one at a time,
    automatically moving all OTHER models off-GPU before each measurement so none of them
    can inflate another's peak-memory reading (fixes the leftover-optimizer/leftover-weights
    artifact: reset_peak_memory_stats() only resets the peak COUNTER, it does not free
    memory already allocated by a previously-trained model still sitting on the GPU).
    Call this ONCE with all models you want compared -- no manual isolation needed, and
    original devices are restored when done. Memory/time depend only on ARCHITECTURE
    (forward-pass shape), not trained weights, so freshly-constructed (untrained) model
    instances give the same numbers as trained ones -- no need to load checkpoints first."""
    results = {}
    original_devices = {name: next(m.parameters()).device for name, m in models_dict.items()}

    for name, model in models_dict.items():
        # evict every other model in the dict so it can't inflate this one's peak reading
        for other_name, other_model in models_dict.items():
            if other_name != name:
                other_model.to('cpu')
        torch.cuda.empty_cache()
        gc.collect()

        model.to(device).eval()

        with torch.no_grad():
            for _ in range(warmup):
                _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()

        # --- memory: reset peak counter only AFTER eviction + warm-up, on an
        #     otherwise-empty GPU, then run one forward pass and read the peak ---
        if device == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)
        with torch.no_grad():
            _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()
        peak_mem = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device == 'cuda' else None

        # --- time: n_runs forward passes, synchronized start/end ---
        if device == 'cuda':
            torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            for _ in range(n_runs):
                _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()
        inf_time = (time.time() - start) / n_runs * 1000  # ms

        results[name] = {
            'Inference Time (ms)': round(inf_time, 4),
            'Peak Memory (MB)': round(peak_mem, 4) if peak_mem is not None else None,
        }
        print(f"[{name}] measured in isolation -> Time: {inf_time:.4f} ms, "
              f"Memory: {peak_mem:.2f} MB" if peak_mem is not None else
              f"[{name}] measured in isolation -> Time: {inf_time:.4f} ms")

    # restore every model to whatever device it was on before this function ran
    for name, model in models_dict.items():
        model.to(original_devices[name])

    return pd.DataFrame(results).T


In [ ]:
# ============================================================
# Per-model checkpointing (Section 8) -- so an interruption (Kaggle/Colab
# disconnect, kernel crash) doesn't require retraining every model from
# scratch. Each model's checkpoint is saved once it finishes its full
# num_epochs run; a re-run loads it back instead of retraining.
# ============================================================
def checkpoint_path(current_dataset, model_name):
    safe_name = model_name.replace(" ", "_").replace("(", "").replace(")", "")
    return f"{OUTPUT_ROOT}/{current_dataset}_{safe_name}_checkpoint.pth"


def save_model_checkpoint(current_dataset, model_name, model, result_dict, history):
    """Called once a model finishes its full num_epochs training run."""
    path = checkpoint_path(current_dataset, model_name)
    torch.save({
        'model_state_dict': model.state_dict(),
        'result': result_dict,
        'history': history,
    }, path)
    print(f"  [checkpoint saved] {path}")


def load_model_checkpoint(current_dataset, model_name, model):
    """Loads weights into `model` in-place and returns (result_dict, history)
    if a completed checkpoint exists for this (dataset, model) pair -- caller
    should then skip training entirely. Returns None otherwise (train normally).
    Respects the global FORCE_RETRAIN flag."""
    path = checkpoint_path(current_dataset, model_name)
    if FORCE_RETRAIN or not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"  [checkpoint found] Skipping training for '{model_name}' -- loaded from {path}")
    return ckpt['result'], ckpt['history']


In [ ]:
class LowRankSparseMultiheadAttention_Visualizable(nn.Module):
    def __init__(self, embed_dim, num_heads, rank, sparsity_ratio=0.5):
        super(LowRankSparseMultiheadAttention_Visualizable, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.rank = rank
        self.sparsity_ratio = sparsity_ratio

        self.q_lows = nn.ModuleList([nn.Linear(embed_dim, rank, bias=False) for _ in range(num_heads)])
        self.q_highs = nn.ModuleList([nn.Linear(rank, embed_dim, bias=False) for _ in range(num_heads)])
        self.k_lows = nn.ModuleList([nn.Linear(embed_dim, rank, bias=False) for _ in range(num_heads)])
        self.k_highs = nn.ModuleList([nn.Linear(rank, embed_dim, bias=False) for _ in range(num_heads)])
        self.v_lows = nn.ModuleList([nn.Linear(embed_dim, rank, bias=False) for _ in range(num_heads)])
        self.v_highs = nn.ModuleList([nn.Linear(rank, embed_dim, bias=False) for _ in range(num_heads)])

        self.out_proj = nn.Linear(embed_dim * num_heads, embed_dim)
        self.scale = embed_dim ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        """Same -inf-before-softmax fix as the non-visualizable class above
        (see its docstring) -- masked_fill instead of multiplying by 0, so
        masked positions get exactly 0 probability after softmax. sparse_mask
        (the boolean keep/drop map) is still returned as-is for visualization."""
        batch_size, num_heads, seq_length, _ = attn_scores.size()
        if seq_length == 1:
            return attn_scores, torch.ones_like(attn_scores)
        num_to_keep = int(sparsity_ratio * seq_length)
        top_scores, _ = torch.topk(attn_scores, k=num_to_keep, dim=-1)
        threshold = top_scores.min(dim=-1, keepdim=True)[0]
        sparse_mask = attn_scores >= threshold
        sparse_attn_scores = attn_scores.masked_fill(~sparse_mask, float('-inf'))
        return sparse_attn_scores, sparse_mask

    def forward(self, x):
        batch_size, seq_length, embed_dim = x.size()

        q, k, v = [], [], []
        for i in range(self.num_heads):
            q.append(self.q_highs[i](self.q_lows[i](x)).view(batch_size, seq_length, embed_dim))
            k.append(self.k_highs[i](self.k_lows[i](x)).view(batch_size, seq_length, embed_dim))
            v.append(self.v_highs[i](self.v_lows[i](x)).view(batch_size, seq_length, embed_dim))

        q = torch.stack(q, dim=1)
        k = torch.stack(k, dim=1)
        v = torch.stack(v, dim=1)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        sparse_attn_scores, sparse_mask = self.sparse_attention(attn_scores, self.sparsity_ratio)
        attn_probs = F.softmax(sparse_attn_scores, dim=-1)
        attn_output = torch.matmul(attn_probs, v)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.num_heads * embed_dim)

        output = self.out_proj(attn_output)
        return output, sparse_attn_scores, sparse_mask


class CustomDeiTLayer_Visualizable(nn.Module):
    def __init__(self, embed_dim, num_heads, rank, mlp_ratio=4., drop_path=0.1, sparsity_ratio=0.5):
        super(CustomDeiTLayer_Visualizable, self).__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LowRankSparseMultiheadAttention_Visualizable(embed_dim, num_heads, rank, sparsity_ratio)
        self.drop_path = nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        attn_output, attn_scores, sparse_mask = self.attn(self.norm1(x))
        x = x + self.drop_path(attn_output)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x, attn_scores, sparse_mask


class HybridStudentModel_Visualizable(nn.Module):
    """Standalone demo model (matches original visualization cell) -- takes pre-computed
    2000-d feature vectors directly (not raw images), for visualizing attention/sparsity."""
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS, rank=GLOBAL_RANK,
                 drop_path_rate=0.1, sparsity_ratio=0.5):
        super(HybridStudentModel_Visualizable, self).__init__()
        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer_Visualizable(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                                          sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(2000, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.deit_embed(x)
        for layer in self.deit_layers:
            x, attn_scores, sparse_mask = layer(x)
        x = self.norm(x)
        x = x.squeeze(1)
        x = self.classifier(x)
        return x, attn_scores, sparse_mask


def visualize_attention_and_sparsity(attn_scores, sparse_mask, head=0):
    attn_scores = attn_scores[0, head].cpu().detach().numpy()
    sparse_mask = sparse_mask[0, head].cpu().detach().numpy()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 5), dpi=150)
    ax1.imshow(attn_scores, cmap='viridis')
    ax1.set_title(f'Attention Scores (Head {head})')
    ax1.set_xlabel('Key Position')
    ax1.set_ylabel('Query Position')

    ax2.imshow(sparse_mask, cmap='gray')
    ax2.set_title(f'Sparsity Mask (Head {head})')
    ax2.set_xlabel('Key Position')
    ax2.set_ylabel('Query Position')
    plt.show()


def forward_with_visualization(model, data_loader, head=1):
    model.eval()
    with torch.no_grad():
        for batch in data_loader:
            images = batch.cuda() if isinstance(batch, torch.Tensor) else batch[0].cuda()
            outputs, attn_scores, sparse_mask = model(images)
            visualize_attention_and_sparsity(attn_scores, sparse_mask, head=head)
            break


# Demo run with dummy random data (as in the original notebook)
demo_batch_size = 32
demo_embed_dim = 768
demo_num_classes = 8  # placeholder: this demo cell uses synthetic random data and is not tied to any dataset in DATASET_CONFIGS, so num_classes (which is only ever set inside run_pipeline) isn't available here

demo_hybrid_model = HybridStudentModel_Visualizable(num_classes=demo_num_classes, embed_dim=demo_embed_dim).cuda()

demo_loader_1 = torch.utils.data.DataLoader(torch.randn(demo_batch_size, 1, 2000), batch_size=demo_batch_size)
forward_with_visualization(demo_hybrid_model, demo_loader_1, head=1)

demo_seq_len = 16  # longer sequence for a more meaningful attention map
demo_loader_2 = torch.utils.data.DataLoader(torch.randn(demo_batch_size, demo_seq_len, 2000), batch_size=demo_batch_size)
forward_with_visualization(demo_hybrid_model, demo_loader_2, head=1)

In [ ]:
# --- Statistical significance analysis: multi-seed runs + paired t-test ---
from scipy import stats

N_SEEDS = 1 if QUICK_TEST_MODE else 3
SEED_LIST = [42, 123, 2024][:N_SEEDS]
STAT_NUM_EPOCHS = GLOBAL_NUM_EPOCHS  # bump this up for the camera-ready run; kept low here to match the rest of the notebook

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_split(dataset_path, seed):
    """Fresh train/val/test split for this seed. Uses a patient-level
    GroupShuffleSplit when the dataset's filenames carry a parseable patient/
    slide ID (e.g. BreakHis) so no patient appears in more than one split;
    falls back to the original image-level random split otherwise (e.g.
    Kather5k, NCT100k -- tile-level benchmarks with no patient metadata)."""
    full_ds = datasets.ImageFolder(dataset_path)
    group_split = patient_group_split(full_ds, seed=seed)

    if group_split is not None:
        train_idx, val_idx, test_idx = group_split
        train_ds = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
        val_ds = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
        test_ds = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)
    else:
        g = torch.Generator().manual_seed(seed)
        n_test = int(len(full_ds) * 0.2)
        n_train = len(full_ds) - n_test
        train_ds, test_ds = random_split(full_ds, [n_train, n_test], generator=g)
        n_val = int(n_train * 0.1)
        n_train2 = n_train - n_val
        train_ds, val_ds = random_split(train_ds, [n_train2, n_val], generator=g)
        train_ds.dataset.transform = train_val_transform
        val_ds.dataset.transform = train_val_transform
        test_ds.dataset.transform = test_transform

    train_ds = quick_subset(train_ds, QUICK_TEST_MAX_TRAIN)
    val_ds = quick_subset(val_ds, QUICK_TEST_MAX_VAL)
    test_ds = quick_subset(test_ds, QUICK_TEST_MAX_TEST)
    tl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    vl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    tel = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    return tl, vl, tel


In [ ]:
# --- Theoretical complexity: LoRa-SMHA vs standard MHA ---
# Notation: E = embed_dim, N = sequence length (tokens), H = heads, R = rank (R << E)
#
# Standard MHA (per layer):
#   Params:  4E^2 + 4E                 (Q,K,V,out projections, each ExE + bias)
#   FLOPs:   O(N*E^2)   for the 4 linear projections
#          + O(N^2*E)   for QK^T and attn*V
#          -----------------------------------------------
#          = O(N*E^2 + N^2*E)
#   Activation memory (attention score matrix): O(H*N^2)
#
# LoRa-SMHA (per layer, RANK-SPACE attention -- Q,K,V never reconstructed to E
# before the attention product; matches the paper's O(N^2 r + N r^2) claim):
#   Params:  4*E*R + R^2               (q_low,k_low,v_low: E->R each, out_proj: R->E,
#                                        rank_mix: RxR)
#   FLOPs:   O(N*E*R)    for the down-projections (Q,K,V) and the final up-projection
#          + O(N^2*R)    for QK^T and attn*V, computed AT RANK R (not E)  <-- was O(N^2*E)
#          + O(N*R^2)    for the r x r inter-head mixing step
#          -----------------------------------------------
#          = O(N*E*R + N^2*R + N*R^2)
#     The N^2*R and N*R^2 terms are the attention block's own scaling cost (what the
#     paper's complexity statement O(N^2 r + N r^2) refers to); N*E*R is the fixed
#     one-time cost of moving data in/out of rank space.
#   Activation memory (attention score matrix): O(H*N^2) -- same shape as standard MHA,
#     just computed on rank-R vectors instead of E-dim vectors, so the matmul itself is
#     cheaper even though the score-matrix SHAPE (N x N per head) is unchanged.

import sympy as sp

E, N, H, R = sp.symbols('E N H R', positive=True, integer=True)

mha_params  = 4*E**2 + 4*E
mha_flops   = 4*N*E**2 + 2*N**2*E          # linear projections + (QK^T + attn*V)

lora_params = 4*E*R + R**2
lora_flops  = 4*N*E*R + 2*N**2*R + N*R**2  # down/up-proj + (QK^T + attn*V at rank R) + rank_mix

print("Standard MHA   params:", mha_params, "   FLOPs:", mha_flops)
print("LoRa-SMHA      params:", lora_params, "   FLOPs:", lora_flops)

reduction_params = sp.simplify(mha_params / lora_params)
print("\nSymbolic params ratio (MHA / LoRa-SMHA):", reduction_params)

E_val, H_val, R_val = 768, GLOBAL_NUM_HEADS, GLOBAL_RANK
rows = []
for N_val in [9, 16, 25, 49, 100]:  # 9 = grid_size=3 (this paper); rest = what-if scaling
    mp = int(mha_params.subs({E: E_val}))
    mf = int(mha_flops.subs({E: E_val, N: N_val}))
    lp = int(lora_params.subs({E: E_val, R: R_val}))
    lf = int(lora_flops.subs({E: E_val, N: N_val, R: R_val}))
    rows.append({
        'N (tokens)': N_val, 'MHA Params': mp, 'MHA FLOPs': mf,
        'LoRa-SMHA Params': lp, 'LoRa-SMHA FLOPs': lf,
        'Param Reduction': f"{mp/lp:.2f}x", 'FLOPs Reduction': f"{mf/lf:.2f}x",
    })
complexity_theory_df = pd.DataFrame(rows)
print(f"\nTable: Theoretical complexity, single attention layer, E={E_val}, H={H_val}, R={R_val}\n")
print(complexity_theory_df.to_string(index=False))


In [ ]:
# --- Empirical (thop-profiled) complexity on the ISOLATED attention layer ---
# Uses the same StandardMHAWrapper / LowRankSparseMultiheadAttention classes already
# defined earlier in the notebook (Cells 15 & 17), at N=9 = grid_size=3 (this paper's setting).

attn_lora = LowRankSparseMultiheadAttention(embed_dim=768, num_heads=GLOBAL_NUM_HEADS, rank=GLOBAL_RANK, sparsity_ratio=0.5).cuda()
attn_mha  = StandardMHAWrapper(embed_dim=768, num_heads=GLOBAL_NUM_HEADS).cuda()

dummy_tokens = torch.randn(1, 9, 768).cuda()  # (B, N=9, E=768)

flops_lora, params_lora = profile(attn_lora, inputs=(dummy_tokens,), verbose=False)
flops_mha,  params_mha  = profile(attn_mha,  inputs=(dummy_tokens,), verbose=False)

time_lora = measure_inference_time(attn_lora, dummy_tokens, n_runs=100)
time_mha  = measure_inference_time(attn_mha, dummy_tokens, n_runs=100)
mem_lora  = measure_memory(attn_lora, dummy_tokens)
mem_mha   = measure_memory(attn_mha, dummy_tokens)

empirical_complexity_df = pd.DataFrame({
    'Standard MHA': {'Params': int(params_mha), 'FLOPs': int(flops_mha),
                      'Time (ms)': round(time_mha, 4), 'Peak Memory (MB)': round(mem_mha, 4)},
    'LoRa-SMHA':    {'Params': int(params_lora), 'FLOPs': int(flops_lora),
                      'Time (ms)': round(time_lora, 4), 'Peak Memory (MB)': round(mem_lora, 4)},
}).T
print(f"Table: Empirical (thop-profiled) complexity, isolated attention layer, N=9, E=768, H={GLOBAL_NUM_HEADS}, R={GLOBAL_RANK}\n")
print(empirical_complexity_df.to_string())


In [ ]:
# --- Plots: empirical bar chart + theoretical FLOPs-vs-N scaling curve ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

x = np.arange(2)
width = 0.35
axes[0].bar(x - width/2, [params_mha, params_lora], width, label='Params')
axes[0].bar(x + width/2, [flops_mha, flops_lora], width, label='FLOPs')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Standard MHA', 'LoRa-SMHA'])
axes[0].set_yscale('log')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Attention-layer Params & FLOPs (N=9, E=768)')
axes[0].legend()

N_range = np.array([9, 16, 25, 49, 100, 196])
mha_flops_vals = [int(mha_flops.subs({E: 768, N: int(n)})) for n in N_range]
lora_flops_vals = [int(lora_flops.subs({E: 768, N: int(n), R: GLOBAL_RANK})) for n in N_range]
axes[1].plot(N_range, mha_flops_vals, marker='o', label='Standard MHA')
axes[1].plot(N_range, lora_flops_vals, marker='s', label=f'LoRa-SMHA (R={GLOBAL_RANK})')
axes[1].axvline(9, color='gray', linestyle='--', alpha=0.5)
axes[1].text(9, max(mha_flops_vals)*0.9, 'this paper\n(grid=3)', fontsize=8, ha='left')
axes[1].set_xlabel('N (tokens)')
axes[1].set_ylabel('FLOPs')
axes[1].set_title('Theoretical FLOPs vs Sequence Length')
axes[1].legend()

plt.tight_layout()
plt.savefig('images/complexity_analysis.png', dpi=150)
plt.show()


In [ ]:
class GradCAM:
    """Standard Grad-CAM applied to a CNN feature-extractor submodule. Works on any
    nn.Module target layer that outputs a (B, C, H, W) spatial map."""
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):
        self.activations = out.detach()

    def _save_grad(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = int(output.argmax(dim=1).item())
        self.model.zero_grad()
        output[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    def remove(self):
        self._fwd.remove()
        self._bwd.remove()


In [ ]:
# --- New model variants needed for structural ablation ---
class FullRankSparseAttention(nn.Module):
    """Ablation: removes the low-rank factorization but KEEPS top-k sparsity,
    to isolate the contribution of low-rank vs sparsity."""
    def __init__(self, embed_dim, num_heads, sparsity_ratio=0.5):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.sparsity_ratio = sparsity_ratio
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.scale = embed_dim ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        """Same -inf-before-softmax fix as LowRankSparseMultiheadAttention
        (see its docstring). This class computes attention at full embed_dim
        (that's the whole point of the 'w/o Low-Rank Factorization' ablation
        -- isolate sparsity from low-rank) -- only the masking mechanism
        changes here, not the attention dimensionality."""
        b, h, t, _ = attn_scores.size()
        if t == 1:
            return attn_scores
        k = max(1, int(sparsity_ratio * t))
        top, _ = torch.topk(attn_scores, k=k, dim=-1)
        thresh = top.min(dim=-1, keepdim=True)[0]
        mask = attn_scores >= thresh
        return attn_scores.masked_fill(~mask, float('-inf'))

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        scores = self.sparse_attention(scores, self.sparsity_ratio)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class SingleBackboneHybridModel(nn.Module):
    """Ablation: keeps the full LoRa-SMHA transformer, removes one of the two CNN branches."""
    def __init__(self, num_classes, backbone='resnet', embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 rank=GLOBAL_RANK, sparsity_ratio=0.5, grid_size=3, drop_path_rate=0.1):
        super().__init__()
        assert backbone in ('resnet', 'densenet')
        if backbone == 'resnet':
            self.backbone = ResNet18_Features()
            in_channels = 512
        else:
            self.backbone = DenseNet121_Features()
            in_channels = 1024
        self.grid_size = grid_size
        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                             sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(in_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        feats = self.backbone(x)
        if self.grid_size != feats.shape[-1]:
            feats = F.adaptive_avg_pool2d(feats, (self.grid_size, self.grid_size))
        b, c, h, w = feats.shape
        feats = feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(feats)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.classifier(x)


### 3.2 Pathology foundation models (UNI / CONCH / Virchow) — R3 fix merged in

This replaces the original Section 18 encoder-loading cell with the **fixed** version from the R3 standalone notebook: CONCH is loaded via its own `conch` package (not `transformers.AutoModel`), and Virchow's raw token-sequence output is pooled into the documented 2560-dim embedding. One-time install needed (internet required):

In [ ]:
!pip install -q git+https://github.com/Mahmoodlab/CONCH.git

In [ ]:
FOUNDATION_MODEL_EPOCHS = GLOBAL_NUM_EPOCHS  # linear-probe epochs; bump up for camera-ready

class _CONCHImageEncoderWrapper(nn.Module):
    """CONCH is NOT a transformers model -- MahmoodLab's own model card loads it via
    their custom `conch` package (open_clip-based), which is why
    `AutoModel.from_pretrained('MahmoodLab/CONCH', trust_remote_code=True)` failed with
    'Unrecognized model ... no model_type key'. This wraps the conch model's
    .encode_image(proj_contrast=False, normalize=False) call -- the pre-projection,
    non-normalized embedding MahmoodLab documents as the correct one for linear probing --
    so it plugs into LinearProbeHead/CLAMAttentionHead exactly like the other encoders."""
    def __init__(self, conch_model):
        super().__init__()
        self.conch_model = conch_model

    def forward(self, x):
        return self.conch_model.encode_image(x, proj_contrast=False, normalize=False)


class _VirchowEncoderWrapper(nn.Module):
    """Virchow's raw timm forward() returns the full token sequence (B, 257, 1280),
    not a flat embedding -- feeding that straight into a Linear(2560, num_classes) is
    what produced 'mat1 and mat2 shapes cannot be multiplied (8224x1280 and 2560x8)'
    (8224 = batch_size * 257 tokens, 1280 = per-token dim, not the 2560 the head expects).
    Per Virchow's model card, the documented 2560-dim embedding is the class token
    concatenated with the mean-pooled patch tokens -- this wrapper does exactly that."""
    def __init__(self, vit_model):
        super().__init__()
        self.vit_model = vit_model

    def forward(self, x):
        output = self.vit_model(x)            # (B, 257, 1280)
        class_token = output[:, 0]             # (B, 1280)
        patch_tokens = output[:, 1:]            # (B, 256, 1280)
        return torch.cat([class_token, patch_tokens.mean(1)], dim=-1)  # (B, 2560)


def load_foundation_encoder(name):
    """Returns (encoder, embed_dim). Encoder is frozen (eval mode, no grad)."""
    if name == 'UNI':
        import timm
        encoder = timm.create_model(
            "hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, num_classes=0
        )
        embed_dim = 1024
    elif name == 'CONCH':
        # One-time setup (needs internet): !pip install -q git+https://github.com/Mahmoodlab/CONCH.git
        from conch.open_clip_custom import create_model_from_pretrained
        conch_model, _ = create_model_from_pretrained(
            'conch_ViT-B-16', "hf_hub:MahmoodLab/conch", hf_auth_token=globals().get('hf_token')
        )
        encoder = _CONCHImageEncoderWrapper(conch_model)
        embed_dim = 512
    elif name == 'Virchow':
        import timm
        vit_model = timm.create_model(
            "hf-hub:paige-ai/Virchow", pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU
        )
        encoder = _VirchowEncoderWrapper(vit_model)
        embed_dim = 2560  # CLS + mean-patch concat per the Virchow paper; adjust if using CLS only
    elif name == 'HIPT':
        # HIPT's patch-level stage is a ViT-S/16 pretrained with DINO on TCGA patches.
        # No official HF Hub weights as of writing -- clone https://github.com/mahmoodlab/HIPT
        # and point this at the local checkpoint:
        #   encoder = torch.hub.load(...); embed_dim = 384
        raise NotImplementedError("Point this at your local HIPT ViT-S/16 checkpoint (see comment above).")
    else:
        raise ValueError(f"Unknown foundation model: {name}")

    encoder = encoder.cuda().eval()
    for p in encoder.parameters():
        p.requires_grad = False
    return encoder, embed_dim


class LinearProbeHead(nn.Module):
    """Frozen encoder + trainable linear classification head -- the standard evaluation
    protocol for foundation-model patch encoders."""
    def __init__(self, encoder, embed_dim, num_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            feats = self.encoder(x)
            if isinstance(feats, (tuple, list)):
                feats = feats[0]
        return self.head(feats)


class CLAMAttentionHead(nn.Module):
    """CLAM-style gated-attention pooling (Lu et al., 2021) adapted as a classification
    head over a single patch's frozen-encoder embedding -- included for architectural
    comparison only, not a faithful multi-instance-learning replication (see markdown note)."""
    def __init__(self, encoder, embed_dim, num_classes, attn_dim=256):
        super().__init__()
        self.encoder = encoder
        self.attn_V = nn.Linear(embed_dim, attn_dim)
        self.attn_U = nn.Linear(embed_dim, attn_dim)
        self.attn_w = nn.Linear(attn_dim, 1)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            feats = self.encoder(x)
            if isinstance(feats, (tuple, list)):
                feats = feats[0]
        a = torch.tanh(self.attn_V(feats)) * torch.sigmoid(self.attn_U(feats))
        gate = torch.sigmoid(self.attn_w(a))
        gated_feats = feats * gate
        return self.classifier(gated_feats)

Hugging Face login (gated models) — run once, before the dataset loop:

In [ ]:
# --- Hugging Face authentication for gated foundation models (UNI/CONCH/Virchow) ---
# One-time setup BEFORE this will work (this is what caused the 401 GatedRepoError above):
#   1. Create/use a Hugging Face account: https://huggingface.co/join
#   2. Request access on EACH gated model page (approval is not always instant --
#      MahmoodLab/paige-ai review requests, so do this well before you need the results):
#        https://huggingface.co/MahmoodLab/uni
#        https://huggingface.co/MahmoodLab/CONCH
#        https://huggingface.co/paige-ai/Virchow
#   3. Create a READ access token: https://huggingface.co/settings/tokens
#   4. Provide the token one of these ways (checked in order below):
#        a. Kaggle: Add-ons -> Secrets -> add a secret named HF_TOKEN
#        b. HF_TOKEN environment variable, if you set one before launching Jupyter
#        c. Already ran `huggingface-cli login` in this environment -- reuses that
#           cached credential, no token needed here at all
#        d. None of the above -- running plain local Jupyter (not Kaggle): this cell
#           will just ask you to paste your token below, right here, no terminal needed.
#           The box hides what you type (like a password field) and nothing is saved
#           to disk by this cell.
#   5. Notebook Settings (right panel, Kaggle only) -> Internet -> ON (downloads need internet)
#
# Once set up, re-run this cell then re-run the Section 18 cells below -- access approval
# can take a while, so if you still see a GatedRepoError immediately after requesting
# access, that's normal; check your HF email/notifications for approval.

from huggingface_hub import login
import getpass

hf_login_ok = False
hf_token = None

# (a) Kaggle Secrets, if we're actually running on Kaggle
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except ModuleNotFoundError:
    pass  # not running on Kaggle -- expected in plain Jupyter, fall through to (b)
except Exception as e:
    print(f"Kaggle Secrets lookup failed ({type(e).__name__}: {e}) -- falling back.")

# (b) HF_TOKEN environment variable
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

# (c) a cached `huggingface-cli login` credential, if one exists
if not hf_token:
    try:
        from huggingface_hub import HfApi
        HfApi().whoami()  # raises if nothing cached
        hf_login_ok = True
        print("Reusing existing cached Hugging Face login (from huggingface-cli login).")
    except Exception:
        pass  # nothing cached -- fall through to (d)

# (d) prompt for a token right here in the notebook (plain Jupyter, no Kaggle/env/cache found)
if not hf_login_ok and not hf_token:
    print("No cached login, HF_TOKEN env var, or Kaggle secret found.")
    print("Paste your Hugging Face token below (from https://huggingface.co/settings/tokens)")
    print("and press Enter -- your typing will be hidden.")
    hf_token = getpass.getpass("HF token: ")

if not hf_login_ok:
    try:
        login(token=hf_token)
        hf_login_ok = True
        print("Hugging Face login successful.")
    except Exception as e:
        print(f"Hugging Face login FAILED: {type(e).__name__}: {e}")
        print("Foundation model downloads (UNI/CONCH/Virchow) below will be skipped until this is fixed --")
        print("see the setup steps in this cell's comments.")


In [ ]:
# --- Verify HF login + gated access, WITHOUT downloading any model weights ---
# Run this any time to check status -- it only fetches metadata (fast), not the
# multi-GB weight files, so it is safe to re-run while waiting on approvals.
from huggingface_hub import HfApi

api = HfApi()

if not hf_login_ok:
    print("Not logged in -- fix the cell above first (check your HF_TOKEN Kaggle Secret).")
else:
    try:
        who = api.whoami()
        print(f"Logged in as: {who['name']}  (email verified: {who.get('email', 'unknown')})\n")
    except Exception as e:
        print(f"Logged in, but whoami() failed: {type(e).__name__}: {e}\n")

    gated_repos = ['MahmoodLab/uni', 'MahmoodLab/CONCH', 'paige-ai/Virchow']
    print("Gated model access status:\n")
    for repo_id in gated_repos:
        try:
            api.model_info(repo_id)
            print(f"  [ACCESS GRANTED]  {repo_id}")
        except Exception as e:
            reason = type(e).__name__
            if 'Gated' in reason or '403' in str(e):
                print(f"  [PENDING/DENIED]  {repo_id}  -- request not yet approved (or was denied)")
            else:
                print(f"  [ERROR]           {repo_id}  -- {reason}: {e}")
    print("\nOnce all three show [ACCESS GRANTED], Section 18 below will download and run for real.")


## 4. `run_pipeline` — the entire per-dataset pipeline (Sections 2, 8–22)

Calling `run_pipeline("CRC7k", DATASET_CONFIGS["CRC7k"]["path"])` runs data loading, teacher-model loading, the main LoRaS-CT vs MHA-Net comparison, every results table/plot, t-SNE, LIME, SHAP, Grad-CAM, the statistical-significance / ablation / foundation-model / k-fold / hyperparameter-ablation / teacher-finetuning / patient-split-audit sections — all labeled with that dataset's name, with images saved under `images/CRC7k/...`.

It returns a dict of the key result tables for that dataset.

In [ ]:
def run_pipeline(CURRENT_DATASET, DATASET_CFG):
    is_presplit = "test_path" in DATASET_CFG
    _path_desc = (f"train_val={DATASET_CFG['train_val_path']} | test={DATASET_CFG['test_path']}"
                  if is_presplit else DATASET_CFG["path"])
    print(f"\n{'#'*80}\n# RUNNING PIPELINE FOR DATASET: {CURRENT_DATASET}\n# Path: {_path_desc}\n{'#'*80}\n")
    img_dir = f"images/{CURRENT_DATASET}"
    os.makedirs(img_dir, exist_ok=True)


    # ======================================================================
    # ## Section 2: Data loading
    # ======================================================================
    # Two shapes supported (see DATASET_CONFIGS in Cell 2):
    #   - flat "path": single folder, one sub-folder per class -- this notebook
    #     does its own 80/10/10-ish train/val/test random_split.
    #   - "train_val_path" + "test_path": dataset ships with its own official
    #     test split (e.g. LC25000) -- the official test set is kept as-is,
    #     and only the train_val folder gets our own 90/10 train/val split.

    if is_presplit:
        # K-fold CV, multi-seed statistical-significance runs, and the patient-split
        # audit further below all resample from `dataset_path` via a fresh
        # ImageFolder() call -- for a pre-split dataset, they should resample from
        # the train_val pool (never the held-out official test set), consistent
        # with the train/val split just above.
        dataset_path = DATASET_CFG["train_val_path"]
        train_val_dataset = datasets.ImageFolder(DATASET_CFG["train_val_path"])
        test_dataset = datasets.ImageFolder(DATASET_CFG["test_path"])

        assert train_val_dataset.classes == test_dataset.classes, (
            f"[{CURRENT_DATASET}] Class mismatch between train/val and test folders: "
            f"{train_val_dataset.classes} vs {test_dataset.classes}. "
            f"Both folders must contain identically-named class sub-folders."
        )
        dataset = train_val_dataset  # so downstream `dataset.classes` references keep working
        num_classes = len(dataset.classes)

        val_size = int(len(train_val_dataset) * 0.1)
        train_size = len(train_val_dataset) - val_size
        train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size])

        train_dataset.dataset.transform = train_val_transform
        val_dataset.dataset.transform = train_val_transform
        test_dataset.transform = test_transform  # plain ImageFolder, not a Subset -- no .dataset indirection
    else:
        dataset_path = DATASET_CFG["path"]
        dataset = datasets.ImageFolder(dataset_path)
        num_classes = len(dataset.classes)

        group_split = patient_group_split(dataset, seed=42)
        if group_split is not None:
            train_idx, val_idx, test_idx = group_split
            print(f"[{CURRENT_DATASET}] Patient/slide IDs detected -- using patient-level "
                  f"GroupShuffleSplit (no patient appears in more than one split).")
            # Separate ImageFolder instances per split (transform baked in at construction) --
            # same pattern already used in the K-Fold section, avoids the shared-underlying-
            # dataset pitfall of mutating .dataset.transform when subsets share one instance.
            train_dataset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
            val_dataset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
            test_dataset = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)
        else:
            # No parseable patient/slide ID (e.g. Kather5k, NCT100k) -- unchanged image-level split.
            test_size = int(len(dataset) * 0.2)
            train_size = len(dataset) - test_size
            train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

            val_size = int(train_size * 0.1)
            train_size = train_size - val_size
            train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

            train_dataset.dataset.transform = train_val_transform
            val_dataset.dataset.transform = train_val_transform
            test_dataset.dataset.transform = test_transform

    # QUICK_TEST_MODE: cap dataset sizes and shrink the batch so a smoke-test run
    # only touches a handful of images instead of the full ~3500-image train set.
    # No-op when QUICK_TEST_MODE is False (see Cell 5).
    train_dataset = quick_subset(train_dataset, QUICK_TEST_MAX_TRAIN)
    val_dataset = quick_subset(val_dataset, QUICK_TEST_MAX_VAL)
    test_dataset = quick_subset(test_dataset, QUICK_TEST_MAX_TEST)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    print(f"[{CURRENT_DATASET}] Classes: {dataset.classes}")
    print(f"[{CURRENT_DATASET}] Train/Val/Test sizes: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}"
          + ("  [QUICK_TEST_MODE: capped]" if QUICK_TEST_MODE else ""))


    # ======================================================================
    # ## Section 3: Teacher models (ViT/DeiT/Swin) -- loaded per dataset (num_classes-dependent)
    # ======================================================================

    teacher_vit = create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_deit = create_model('deit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_swin = create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=num_classes).cuda()

    teacher_models = [teacher_vit, teacher_deit, teacher_swin]

    import torch

    torch.save(teacher_vit.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_teacher_vit.pth')
    torch.save(teacher_deit.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_teacher_deit.pth')
    torch.save(teacher_swin.state_dict(), f'{OUTPUT_ROOT}/{CURRENT_DATASET}_teacher_swin.pth')

    os.system(f"ls -la {OUTPUT_ROOT}/{CURRENT_DATASET}_teacher_*.pth 2>/dev/null || echo 'no checkpoints yet'")


    # ======================================================================
    # ## Section 8: Run the main comparison -- LoRaS-CT vs MHA-Net baseline
    # ======================================================================

    def run_full_comparison(num_classes, teacher_models, train_loader, val_loader, test_loader,
                             num_epochs=20, device='cuda', alpha=0.5, temperature=3.0,
                             lr=0.01, momentum=0.9, grid_size=3):

        models_to_compare = {
            # MHA-Net intentionally does NOT read GLOBAL_NUM_HEADS/GLOBAL_NUM_LAYERS here --
            # those were tuned via grid search for LoRaS-CT ONLY (MHA-Net was dropped from
            # that sweep). Sharing the same globals meant MHA-Net silently inherited
            # LoRaS-CT's tuned config (e.g. heads=2), which is a config MHA-Net was never
            # validated on -- an unfair/accidental comparison. MHA-Net now always uses its
            # own standard baseline config (heads=8, layers=2) regardless of what the
            # globals are set to for LoRaS-CT.
            'MHA-Net (baseline)': MHANetBaseline(num_classes, num_heads=8, num_layers=2, grid_size=grid_size).to(device),
            'LoRaS-CT': HybridStudentModel(num_classes, grid_size=grid_size).to(device),
        }

        results = {}
        trained_models = {}

        for name, model in models_to_compare.items():
            print(f"\n{'='*60}")
            print(f"Training: {name}  (grid_size={grid_size} -> N={grid_size*grid_size} tokens)")
            print(f"{'='*60}")

            cached = load_model_checkpoint(CURRENT_DATASET, name, model)
            if cached is not None:
                results[name], cached_history = cached
                trained_models[name] = {'model': model, **cached_history}
                continue

            criterion = DistillationLoss(alpha=alpha, temperature=temperature)
            optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)

            trained_model, train_losses, val_losses, train_accs, val_accs = \
                train_model_with_distillation(
                    model, teacher_models, train_loader, val_loader,
                    criterion, optimizer, num_epochs=num_epochs
                )

            test_accuracy, test_loss = evaluate_model(trained_model, test_loader, criterion.ce_loss)
            print(f"[{name}] Test Accuracy: {test_accuracy:.2f}%, Test Loss: {test_loss:.4f}")

            dummy_input = torch.randn(1, 3, 224, 224).to(device)
            params = count_parameters(trained_model)
            flops, _ = profile(trained_model, inputs=(dummy_input,), verbose=False)
            inf_time = measure_inference_time(trained_model, dummy_input, device=device)
            memory = measure_memory(trained_model, dummy_input, device=device)

            results[name] = {
                'Test Accuracy (%)': test_accuracy,
                'Params (M)': params / 1e6,
                'FLOPs (G)': flops / 1e9,
                'Inference Time (ms)': inf_time,
                'Peak Memory (MB)': memory,
            }
            history = {'train_losses': train_losses, 'val_losses': val_losses,
                       'train_accs': train_accs, 'val_accs': val_accs}
            trained_models[name] = {'model': trained_model, **history}

            # Checkpoint now that this model's full num_epochs run has finished --
            # a re-run after an interruption skips straight past this model.
            save_model_checkpoint(CURRENT_DATASET, name, trained_model, results[name], history)

        print(f"\n{'='*80}")
        print(f"FINAL COMPARISON -- {CURRENT_DATASET}")
        print(f"{'='*80}")
        metrics = ['Test Accuracy (%)', 'Params (M)', 'FLOPs (G)', 'Inference Time (ms)', 'Peak Memory (MB)']
        header = f"{'Metric':<25}" + "".join(f"{name:<25}" for name in results.keys())
        print(header)
        for metric in metrics:
            row = f"{metric:<25}"
            for name in results.keys():
                row += f"{results[name][metric]:<25.4f}"
            print(row)

        return results, trained_models


    # grid_size=3 -> 9 tokens, matches the architecture diagram
    # (use grid_size=7 for 49 tokens / native CNN resolution, as in the alternate original cell)
    comparison_results, comparison_trained_models = run_full_comparison(
        num_classes=num_classes,
        teacher_models=teacher_models,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        num_epochs=GLOBAL_NUM_EPOCHS,
        device='cuda',
        grid_size=3,
    )

    # Convenience handles used by the "results/plots" section further down
    loras_ct_model = comparison_trained_models['LoRaS-CT']['model']
    mha_net_model = comparison_trained_models['MHA-Net (baseline)']['model']

    print(f'LoRaS-CT (Algorithm-1-accurate) : {count_parameters(loras_ct_model):,} parameters')
    print(f'MHA-Net (baseline)         : {count_parameters(mha_net_model):,} parameters')
    print(f'Teacher ViT                : {count_parameters(teacher_vit):,} parameters')
    print(f'Teacher DeiT                : {count_parameters(teacher_deit):,} parameters')
    print(f'Teacher Swin Transformer   : {count_parameters(teacher_swin):,} parameters')


    # ======================================================================
    # ## Section 8b: Corrected efficiency measurement (feeds Table 5 / Table 10)
    # ------------------------------------------------------------------------
    # MHA-Net and LoRaS-CT were trained back-to-back above with nothing freed
    # between them (optimizer + weights of the first model still resident when
    # the second was profiled) -- that made the earlier Peak Memory numbers a
    # measurement artifact, not a real architectural difference. This section
    # re-measures Inference Time + Peak Memory for MHA-Net, LoRaS-CT, and the
    # three efficient-attention baselines (Linformer/Performer/BigBird) all in
    # ISOLATION (see measure_efficiency_isolated, Cell 15) and OVERWRITES the
    # entries in comparison_results, so every downstream table (Table 5, Table 10,
    # Section 9c) reads the corrected numbers automatically -- single source of
    # truth, no separate manual runs needed.
    # ======================================================================
    _eff_embed_dim, _eff_num_heads, _eff_num_layers, _eff_grid_size = 768, GLOBAL_NUM_HEADS, GLOBAL_NUM_LAYERS, 3
    _eff_N = _eff_grid_size * _eff_grid_size
    _eff_dummy = torch.randn(1, 3, 224, 224).to('cuda')

    # NOTE: built on CPU deliberately -- do NOT chain .cuda() here. If all 5 models
    # land on GPU at once (which .cuda() here would do, on top of mha_net_model and
    # loras_ct_model already being resident), that's 5 models resident simultaneously
    # BEFORE measure_efficiency_isolated() ever gets a chance to start evicting --
    # defeats the isolation and can OOM on smaller GPUs. Let the function move each
    # model to GPU only when it's actually being measured (it already does this).
    mha_net_model.to('cpu')
    loras_ct_model.to('cpu')
    torch.cuda.empty_cache()
    gc.collect()

    _efficiency_models = {
        'MHA-Net (baseline)': mha_net_model,
        'LoRaS-CT': loras_ct_model,
        'Linformer': GenericHybridModel(
            num_classes, attn_factory=lambda: LinformerAttention(_eff_embed_dim, _eff_num_heads, seq_len=_eff_N),
            embed_dim=_eff_embed_dim, num_layers=_eff_num_layers, grid_size=_eff_grid_size),
        'Performer': GenericHybridModel(
            num_classes, attn_factory=lambda: PerformerAttention(_eff_embed_dim, _eff_num_heads),
            embed_dim=_eff_embed_dim, num_layers=_eff_num_layers, grid_size=_eff_grid_size),
        'BigBird': GenericHybridModel(
            num_classes, attn_factory=lambda: BigBirdAttention(_eff_embed_dim, _eff_num_heads, block_size=64),
            embed_dim=_eff_embed_dim, num_layers=_eff_num_layers, grid_size=_eff_grid_size),
    }

    print(f"\n{'='*70}\nSection 8b: isolated efficiency measurement -- {CURRENT_DATASET}\n{'='*70}")
    isolated_efficiency_table = measure_efficiency_isolated(
        _efficiency_models, _eff_dummy, device='cuda', n_runs=100
    )
    print(isolated_efficiency_table.to_string())

    # Overwrite the co-resident (potentially inflated) memory/time entries in
    # comparison_results with the corrected isolated ones. Table 5 (Section 13b)
    # reads straight from comparison_results, so the fix propagates automatically
    # and there is still exactly ONE number per metric per model in the notebook.
    for _mname in ['MHA-Net (baseline)', 'LoRaS-CT']:
        comparison_results[_mname]['Inference Time (ms)'] = isolated_efficiency_table.loc[_mname, 'Inference Time (ms)']
        comparison_results[_mname]['Peak Memory (MB)'] = isolated_efficiency_table.loc[_mname, 'Peak Memory (MB)']

    # free the 3 measurement-only baseline instances (Linformer/Performer/BigBird
    # get freshly re-built and actually trained later in Section 9c -- these copies
    # were only for the isolated efficiency measurement above)
    del _efficiency_models
    torch.cuda.empty_cache()
    gc.collect()

    # measure_efficiency_isolated() restores each model to the device it captured
    # at function-start -- which was 'cpu' for these two (we moved them there on
    # purpose above). Move them back to GPU now since Section 9 below expects
    # mha_net_model / loras_ct_model resident on GPU for plotting/evaluation.
    mha_net_model.to('cuda')
    loras_ct_model.to('cuda')


    # ======================================================================
    # ## Section 9: Results & plots
    # ======================================================================

    def get_predictions_and_labels(model, data_loader, input_size=None):
        # Same input_size=None convention as _train_plain_classifier above --
        # default leaves every other caller (LoRaS-CT, attention baselines, the
        # other 7 SOTA backbones) completely unaffected.
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images, labels = images.cuda(), labels.cuda()
                if input_size is not None:
                    images = F.interpolate(images, size=input_size, mode='bilinear', align_corners=False)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return np.array(all_preds), np.array(all_labels)


    def get_predictions_and_labels_probs(model, data_loader):
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images, labels = images.cuda(), labels.cuda()
                outputs = model(images)
                all_preds.append(outputs.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        return np.vstack(all_preds), np.concatenate(all_labels)


    def plot_confusion_matrix(y_true, y_pred, class_names, save_path="images/confusion_matrix.png"):
        # FIX: without labels=, confusion_matrix() infers the class count from whatever
        # actually appears in y_true/y_pred -- with a small sample (e.g. QUICK_TEST_MODE),
        # a rare class may not appear at all, shrinking the matrix and breaking the mapping
        # to class_names. Passing labels= explicitly forces the full, correctly-sized matrix
        # (unseen classes just show up as all-zero rows/columns).
        cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

        plt.figure(figsize=(6, 6), dpi=300)
        disp.plot(cmap=plt.cm.Blues, ax=plt.gca(), values_format='d', colorbar=False)
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                plt.text(j, i, f'{cm[i, j]}', ha='center', va='center', fontsize=10, color='black',
                         fontweight='bold', bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.4'))
        plt.xticks(rotation=45, ha='right', fontsize=12)
        plt.yticks(fontsize=12)
        plt.xlabel('Predicted Labels', fontsize=12)
        plt.ylabel('True Labels', fontsize=12)
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
        plt.show()

        # FIX (same root cause as the confusion_matrix() call above): with a small sample
        # (e.g. QUICK_TEST_MODE), a class may have zero predicted or zero true samples,
        # which (a) makes precision/recall "ill-defined" for that class -- zero_division=0
        # silences the warning and reports 0 for that class rather than crashing, and
        # (b) makes classification_report() infer fewer classes than len(class_names) --
        # labels= forces it to report all classes (missing ones just show 0s), matching
        # the length of target_names.
        all_labels = list(range(len(class_names)))
        print("Accuracy:", accuracy_score(y_true, y_pred))
        print("Precision:", precision_score(y_true, y_pred, average='weighted', labels=all_labels, zero_division=0))
        print("Recall:", recall_score(y_true, y_pred, average='weighted', labels=all_labels, zero_division=0))
        print("F1 Score:", f1_score(y_true, y_pred, average='weighted', labels=all_labels, zero_division=0))
        print("\nClassification Report:\n", classification_report(
            y_true, y_pred, labels=all_labels, target_names=class_names, zero_division=0))


    def plot_roc_curve(y_true, y_scores, class_names, save_path="images/roc_curve.png"):
        n_classes = len(class_names)
        y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))
        # sklearn's label_binarize collapses to a single column (N,1) instead of
        # one column per class whenever there are exactly 2 classes (its normal
        # binary-classification convention) -- this is what caused the previous
        # IndexError when the loop below tried to index column 1. Datasets vary
        # in class count (some genuinely have 2 classes, e.g. benign/malignant),
        # so this is handled generically rather than assumed away.
        if n_classes == 2 and y_true_bin.shape[1] == 1:
            y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])
        plt.figure(figsize=(6, 5), dpi=300)
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, lw=3, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate', fontsize=12)
        plt.ylabel('True Positive Rate', fontsize=12)
        plt.title('Receiver Operating Characteristic')
        plt.legend(loc='lower right', fontsize=8)
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
        plt.show()


    def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies,
                              save_path="images/training_curves.png"):
        epochs = range(1, len(train_losses) + 1)
        plt.figure(figsize=(10, 4), dpi=300)

        plt.subplot(1, 2, 1)
        plt.plot(epochs, train_losses, label='Train Loss', color='blue', lw=3)
        plt.plot(epochs, val_losses, label='Validation Loss', color='orange', lw=3)
        plt.title('Training and Validation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend(fontsize=10)

        plt.subplot(1, 2, 2)
        plt.plot(epochs, train_accuracies, label='Train Accuracy', color='green', lw=3)
        plt.plot(epochs, val_accuracies, label='Validation Accuracy', color='red', lw=3)
        plt.title('Training and Validation Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy (%)')
        plt.legend(fontsize=10)

        plt.tight_layout()
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
        plt.show()

    # ---- Choose which trained model to evaluate/plot here ----
    model_to_evaluate = loras_ct_model          # options: loras_ct_model, mha_net_model
    model_to_evaluate_name = 'LoRaS-CT'         # must match a key in comparison_results / comparison_trained_models
    train_losses_to_plot = comparison_trained_models[model_to_evaluate_name]['train_losses']
    val_losses_to_plot = comparison_trained_models[model_to_evaluate_name]['val_losses']
    train_accs_to_plot = comparison_trained_models[model_to_evaluate_name]['train_accs']
    val_accs_to_plot = comparison_trained_models[model_to_evaluate_name]['val_accs']
    # ------------------------------------------------------------

    test_predictions, test_labels = get_predictions_and_labels(model_to_evaluate, test_loader)
    plot_confusion_matrix(test_labels, test_predictions, dataset.classes,
                           save_path=f"{img_dir}/confusion_matrix.png")

    test_probs, test_labels_probs = get_predictions_and_labels_probs(model_to_evaluate, test_loader)
    plot_roc_curve(test_labels_probs, test_probs, dataset.classes,
                    save_path=f"{img_dir}/roc_curve.png")

    plot_training_curves(train_losses_to_plot, val_losses_to_plot, train_accs_to_plot, val_accs_to_plot,
                          save_path=f"{img_dir}/training_curves.png")

    # Pulled from comparison_results (Section 8) instead of re-measuring, so this can never
    # mismatch the FINAL COMPARISON table or Table 5.
    print(f"Peak Memory Usage (MB): {comparison_results[model_to_evaluate_name]['Peak Memory (MB)']:.2f}")
    print(f"Per-image Inference Time (ms): {comparison_results[model_to_evaluate_name]['Inference Time (ms)']:.4f}")

    print(f"Number of training images: {len(train_loader.dataset)}")
    print(f"Number of validation images: {len(val_loader.dataset)}")
    print(f"Number of test images: {len(test_loader.dataset)}")

    if hasattr(model_to_evaluate, 'resnet') and hasattr(model_to_evaluate, 'densenet'):
        custom_params = count_custom_parameters(model_to_evaluate, exclude_list=[model_to_evaluate.resnet, model_to_evaluate.densenet])
        print(f'Number of trainable parameters in custom (non-backbone) layers: {custom_params}')


    # ======================================================================
    # ## Section 9b: Table 10 -- attention-mechanism efficiency comparison
    # ======================================================================

    # ============================================================
    # Table 10: efficient-attention baseline comparison
    # ============================================================
    embed_dim, num_heads, num_layers = 768, GLOBAL_NUM_HEADS, GLOBAL_NUM_LAYERS
    grid_size = 3
    N = grid_size * grid_size        # real sequence length in this architecture (=9, not 1)
    rank = GLOBAL_RANK               # matches LowRankSparseMultiheadAttention's default

    def count_params(module):
        return sum(p.numel() for p in module.parameters() if p.requires_grad)

    attn_factories = {
        'Linformer':               lambda: LinformerAttention(embed_dim, num_heads, seq_len=N),
        'Performer':                lambda: PerformerAttention(embed_dim, num_heads),
        'BigBird':                  lambda: BigBirdAttention(embed_dim, num_heads, block_size=64),
        'Standard MHA (MHA-Net)':   lambda: StandardMHAWrapper(embed_dim, num_heads),
        'LoRa-SMHA (proposed)':     lambda: LowRankSparseMultiheadAttention(embed_dim, num_heads, rank=rank),
    }

    dummy_img = torch.randn(1, 3, 224, 224)
    dummy_seq = torch.randn(1, N, embed_dim)

    table10_rows = {}
    for name, factory in attn_factories.items():
        attn_module = factory()
        attn_params = count_params(attn_module)
        attn_flops, _ = profile(attn_module, inputs=(dummy_seq,), verbose=False)

        full_model = GenericHybridModel(
            num_classes, attn_factory=factory, embed_dim=embed_dim,
            num_layers=num_layers, grid_size=grid_size
        )
        full_params = count_params(full_model)
        full_flops, _ = profile(full_model, inputs=(dummy_img,), verbose=False)

        table10_rows[name] = {
            'Attn. Params': attn_params,
            'Attn. FLOPs': attn_flops,
            'Full-Model Params': full_params,
            'Full-Model FLOPs': full_flops,
        }

    table10 = pd.DataFrame(table10_rows).T[
        ['Attn. Params', 'Attn. FLOPs', 'Full-Model Params', 'Full-Model FLOPs']
    ]

    # Merge in the ISOLATED Inference Time / Peak Memory numbers from Section 8b
    # (measured with every other model evicted from GPU, so nothing here is
    # inflated by another model's leftover weights/optimizer state). Table 10's
    # row names differ slightly from Section 8b's dict keys, hence the mapping.
    _table10_name_map = {
        'Standard MHA (MHA-Net)': 'MHA-Net (baseline)',
        'LoRa-SMHA (proposed)': 'LoRaS-CT',
        'Linformer': 'Linformer',
        'Performer': 'Performer',
        'BigBird': 'BigBird',
    }
    table10['Inference Time (ms)'] = [
        isolated_efficiency_table.loc[_table10_name_map[n], 'Inference Time (ms)'] for n in table10.index
    ]
    table10['Peak Memory (MB)'] = [
        isolated_efficiency_table.loc[_table10_name_map[n], 'Peak Memory (MB)'] for n in table10.index
    ]

    baselines = ['Linformer', 'Performer', 'BigBird', 'Standard MHA (MHA-Net)']
    best_attn = table10.loc[baselines, 'Attn. Params'].min()
    best_flops = table10.loc[baselines, 'Attn. FLOPs'].min()
    best_full_p = table10.loc[baselines, 'Full-Model Params'].min()
    best_full_f = table10.loc[baselines, 'Full-Model FLOPs'].min()

    proposed = table10.loc['LoRa-SMHA (proposed)']
    reduction = pd.Series({
        'Attn. Params':       f"{(1 - proposed['Attn. Params'] / best_attn) * 100:.1f}%",
        'Attn. FLOPs':        f"{(1 - proposed['Attn. FLOPs'] / best_flops) * 100:.1f}%",
        'Full-Model Params':  f"{(1 - proposed['Full-Model Params'] / best_full_p) * 100:.1f}%",
        'Full-Model FLOPs':   f"{(1 - proposed['Full-Model FLOPs'] / best_full_f) * 100:.1f}%",
    }, name='Reduction vs. best baseline')

    print(f"Table 10: attention-mechanism efficiency comparison -- {CURRENT_DATASET} (N={N} tokens, embed_dim={embed_dim}, heads={num_heads})\n")
    print(table10.round(0).to_string())
    print()
    print(reduction.to_string())


    # ======================================================================
    # ## Section 9c: Trained comparison -- efficient-attention baselines
    #    (Linformer / Performer / BigBird / Standard MHA) + 8 SOTA transformer
    #    backbones, vs LoRaS-CT (9 models total in the final table, matching
    #    Fig. 4's model set).
    #    LoRaS-CT is NOT retrained here: its accuracy/precision/recall/F1 are
    #    reused directly from Section 8's comparison_results and the
    #    test_predictions/test_labels already computed for loras_ct_model in
    #    Section 9 above.
    # ======================================================================
    print(f"\n{'='*60}\nSection 9c: Trained baseline + SOTA comparison [{CURRENT_DATASET}]\n{'='*60}")

    def _compute_prf(labels, preds):
        return (
            precision_score(labels, preds, average='weighted', zero_division=0),
            recall_score(labels, preds, average='weighted', zero_division=0),
            f1_score(labels, preds, average='weighted', zero_division=0),
        )

    sota_comparison_rows = {}

    # ---- 9c.1: reuse LoRaS-CT's already-trained results (no retraining) ----
    _lora_prec, _lora_rec, _lora_f1 = _compute_prf(test_labels, test_predictions)
    sota_comparison_rows['LoRaS-CT'] = {
        'Accuracy (%)': round(comparison_results['LoRaS-CT']['Test Accuracy (%)'], 2),
        'Precision': round(_lora_prec, 4),
        'Recall': round(_lora_rec, 4),
        'F1-score': round(_lora_f1, 4),
    }

    # ---- 9c.2: train the 4 efficient-attention baselines (same multi-teacher
    #    distillation recipe as Section 8, same CNN backbone/embed/pooling as
    #    LoRaS-CT -- isolates the attention mechanism as the only variable) ----
    attn_baseline_factories_trained = {
        'Linformer':               lambda: LinformerAttention(embed_dim, num_heads, seq_len=N),
        'Performer':                lambda: PerformerAttention(embed_dim, num_heads),
        'BigBird':                  lambda: BigBirdAttention(embed_dim, num_heads, block_size=64),
        'MHA-Net (Standard MHA)':   lambda: StandardMHAWrapper(embed_dim, num_heads),
    }

    for _name, _factory in attn_baseline_factories_trained.items():
        print(f"\n--- Training attention baseline: {_name} [{CURRENT_DATASET}] ---")
        _m = GenericHybridModel(num_classes, attn_factory=_factory, embed_dim=embed_dim,
                                 num_layers=num_layers, grid_size=grid_size).cuda()
        _crit = DistillationLoss(alpha=0.5, temperature=3.0)
        _opt = optim.SGD(_m.parameters(), lr=0.01, momentum=0.9)
        train_model_with_distillation(_m, teacher_models, train_loader, val_loader,
                                       _crit, _opt, num_epochs=GLOBAL_NUM_EPOCHS)
        _acc, _ = evaluate_model(_m, test_loader, _crit.ce_loss)
        _preds, _labels = get_predictions_and_labels(_m, test_loader)
        _prec, _rec, _f1 = _compute_prf(_labels, _preds)
        sota_comparison_rows[_name] = {
            'Accuracy (%)': round(_acc, 2), 'Precision': round(_prec, 4),
            'Recall': round(_rec, 4), 'F1-score': round(_f1, 4),
        }
        _m.cpu(); torch.cuda.empty_cache()

    # ---- 9c.3: fine-tune the 8 SOTA transformer backbones (standalone
    #    classifiers, standard supervised fine-tuning -- these ARE the model
    #    families the distillation teachers are drawn from, so distillation
    #    doesn't apply here; matches how Fig. 4's baselines were produced) ----
    SOTA_BACKBONES = {
        'DeiT B16':  'deit_base_patch16_224',
        'ViT B16':   'vit_base_patch16_224',
        'ViT L16':   'vit_large_patch16_224',
        'ViT L32':   'vit_large_patch32_224',
        'ViT B32':   'vit_base_patch32_224',
        'Swin B4':   'swin_base_patch4_window7_224',
        'Swin V2 S': 'swinv2_small_window16_256',
        'Swin L4':   'swin_large_patch4_window7_224',
    }
    if QUICK_TEST_MODE:
        # Smoke-test only 2 of the 8 SOTA backbones -- full 8-way fine-tuning is
        # multi-hour even at GLOBAL_NUM_EPOCHS=10; QUICK_TEST_MODE just needs to
        # verify the loop RUNS without crashing, same philosophy as Cell 5.
        SOTA_BACKBONES = dict(list(SOTA_BACKBONES.items())[:2])
        print("QUICK_TEST_MODE: only smoke-testing 2/8 SOTA backbones.")

    def _train_plain_classifier(model, train_loader, val_loader, num_epochs, lr=1e-4, input_size=None):
        # input_size=None (default) -> images used exactly as the shared 224x224
        # loaders provide them, for every backbone. Only backbones that explicitly
        # opt in (e.g. Swin V2 S needing 256x256) pass a value here; the shared
        # train/val/test loaders themselves are never modified.
        model = model.cuda()
        optimizer = optim.AdamW(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        for _epoch in range(num_epochs):
            model.train()
            for images, labels in train_loader:
                images, labels = images.cuda(), labels.cuda()
                if input_size is not None:
                    images = F.interpolate(images, size=input_size, mode='bilinear', align_corners=False)
                optimizer.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward()
                optimizer.step()
        return model

    for _display_name, _timm_name in SOTA_BACKBONES.items():
        print(f"\n--- Fine-tuning SOTA backbone: {_display_name} ({_timm_name}) [{CURRENT_DATASET}] ---")
        try:
            _sota_model = create_model(_timm_name, pretrained=True, num_classes=num_classes)
        except Exception as _e:
            print(f"  Skipping {_display_name}: could not load '{_timm_name}' ({_e})")
            continue
        # NOTE: our data pipeline is fixed at 224x224. Most timm backbones above
        # accept that (or are resolution-flexible), but a few (e.g. Swin V2 S,
        # 'swinv2_small_window16_256') hard-assert their trained input resolution
        # internally and will raise AssertionError on the very first forward pass
        # even though loading succeeded. Skip those rather than aborting the whole
        # pipeline -- same "skip and continue" philosophy as the load-failure case
        # right above.
        # Swin V2 S ('swinv2_small_window16_256') internally asserts its trained
        # input resolution (256x256) and will raise on a forward pass fed 224x224
        # images. Resize ONLY for this model, on-the-fly, inside the helper
        # functions -- the shared train/val/test loaders stay 224x224 for every
        # other backbone (and every other model in the whole notebook).
        _backbone_input_size = 256 if _timm_name == 'swinv2_small_window16_256' else None
        try:
            _sota_model = _train_plain_classifier(_sota_model, train_loader, val_loader,
                                                    GLOBAL_NUM_EPOCHS, input_size=_backbone_input_size)
            _preds, _labels = get_predictions_and_labels(_sota_model, test_loader,
                                                            input_size=_backbone_input_size)
            _acc = accuracy_score(_labels, _preds) * 100
            _prec, _rec, _f1 = _compute_prf(_labels, _preds)
            sota_comparison_rows[_display_name] = {
                'Accuracy (%)': round(_acc, 2), 'Precision': round(_prec, 4),
                'Recall': round(_rec, 4), 'F1-score': round(_f1, 4),
            }
        except Exception as _e:
            print(f"  Skipping {_display_name}: {_e}")
        finally:
            _sota_model.cpu(); del _sota_model; torch.cuda.empty_cache()

    sota_comparison_df = pd.DataFrame(sota_comparison_rows).T[
        ['Accuracy (%)', 'Precision', 'Recall', 'F1-score']
    ]
    print(f"\nSection 9c comparison table -- {CURRENT_DATASET} "
          f"(efficient-attention baselines + SOTA backbones vs LoRaS-CT, reused)\n")
    print(sota_comparison_df.to_string())

    # ---- Fig. 4 style plot: Precision & Recall across every model ----
    _fig, _ax = plt.subplots(figsize=(10, 6))
    _model_order = list(sota_comparison_df.index)
    _ax.plot(_model_order, sota_comparison_df['Precision'], marker='o', label='Precision')
    _ax.plot(_model_order, sota_comparison_df['Recall'], marker='s', linestyle='--', label='Recall')
    _ax.set_xlabel('Models'); _ax.set_ylabel('Score')
    _ax.set_title(f'Precision and Recall across baseline models -- {CURRENT_DATASET}')
    plt.xticks(rotation=45, ha='right')
    _ax.legend(); plt.tight_layout()
    plt.savefig(f"{img_dir}/sota_precision_recall_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()


    # ======================================================================
    # ## Section 10: t-SNE visualizations
    # ======================================================================

    def extract_pooled_features(model, data_loader):
        """Extract flattened features + labels from a feature-extractor model."""
        model.eval()
        all_features, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images = images.cuda()
                features = model(images)
                features = features.view(features.size(0), -1)
                all_features.append(features.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return np.vstack(all_features), np.array(all_labels)


    def plot_tsne(features, labels, title, class_names):
        # FIX: sklearn requires perplexity < n_samples; the default perplexity=30 crashes
        # on small samples (e.g. QUICK_TEST_MODE's capped test_loader). Scale it down safely.
        n_samples = features.shape[0]
        perplexity = min(30, max(n_samples - 1, 1))
        tsne = TSNE(n_components=2, random_state=0, perplexity=perplexity)
        features_2d = tsne.fit_transform(features)

        plt.figure(figsize=(6, 6))
        scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=labels,
                               cmap=plt.get_cmap('tab10', len(class_names)), alpha=0.7)
        plt.xlabel('t-SNE Component 1')
        plt.ylabel('t-SNE Component 2')
        plt.title(title)
        plt.legend(handles=scatter.legend_elements()[0], labels=class_names, loc='best')
        plt.savefig(f'{img_dir}/{title.lower().replace(" ", "_")}.png', dpi=300)
        plt.show()


    # ResNet18 / DenseNet121 features via a pooled (avgpool) view, for t-SNE
    class ResNet18_PooledFeatures(nn.Module):
        def __init__(self):
            super().__init__()
            resnet = models.resnet18(pretrained=True)
            self.model = nn.Sequential(*list(resnet.children())[:-1])  # keep avgpool, drop fc

        def forward(self, x):
            return self.model(x)


    class DenseNet121_PooledFeatures(nn.Module):
        def __init__(self):
            super().__init__()
            densenet = models.densenet121(pretrained=True)
            self.features = densenet.features  # conv/dense blocks only, no classifier

        def forward(self, x):
            x = self.features(x)
            x = F.relu(x, inplace=False)
            x = F.adaptive_avg_pool2d(x, (1, 1))
            return x  # (B, 1024, 1, 1) -- pools to (B, 1024) after .view() in extract_pooled_features


    resnet_pooled_model = ResNet18_PooledFeatures().cuda()
    densenet_pooled_model = DenseNet121_PooledFeatures().cuda()

    resnet_tsne_features, resnet_tsne_labels = extract_pooled_features(resnet_pooled_model, test_loader)
    densenet_tsne_features, densenet_tsne_labels = extract_pooled_features(densenet_pooled_model, test_loader)
    combined_tsne_features = np.concatenate((resnet_tsne_features, densenet_tsne_features), axis=1)

    print("ResNet features shape:", resnet_tsne_features.shape)
    print("DenseNet features shape:", densenet_tsne_features.shape)
    print("Combined features shape:", combined_tsne_features.shape)

    plot_tsne(resnet_tsne_features, resnet_tsne_labels, f'{CURRENT_DATASET} - t-SNE Plot of ResNet18 Features', dataset.classes)
    plot_tsne(densenet_tsne_features, densenet_tsne_labels, f'{CURRENT_DATASET} - t-SNE Plot of DenseNet121 Features', dataset.classes)
    plot_tsne(combined_tsne_features, resnet_tsne_labels, f'{CURRENT_DATASET} - t-SNE Plot of Combined Features', dataset.classes)

    # t-SNE on the LoRaS-CT model's internal (pre-classifier, pooled) features.
    def extract_hybrid_features_for_tsne(model, data_loader):
        model.eval()
        all_features, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images = images.cuda()
                _, feats = model(images, return_features=True)
                feats = feats.view(feats.size(0), -1)
                all_features.append(feats.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return np.vstack(all_features), np.array(all_labels)


    hybrid_tsne_features, hybrid_tsne_labels = extract_hybrid_features_for_tsne(loras_ct_model, test_loader)

    # Same perplexity fix as Cell 29's plot_tsne() -- see comment there.
    n_samples = hybrid_tsne_features.shape[0]
    perplexity = min(30, max(n_samples - 1, 1))
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
    hybrid_tsne_2d = tsne.fit_transform(hybrid_tsne_features)

    df_tsne = pd.DataFrame(data=hybrid_tsne_2d, columns=['Dim1', 'Dim2'])
    df_tsne['Label'] = hybrid_tsne_labels

    plt.figure(figsize=(10, 8))
    sns.scatterplot(x='Dim1', y='Dim2', hue='Label', data=df_tsne, palette='tab10', marker='o', alpha=0.7)
    plt.title(f'{CURRENT_DATASET} - t-SNE Visualization of Hybrid Model Features')
    plt.xlabel('Dimension 1')
    plt.ylabel('Dimension 2')
    plt.legend(title='Classes', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.savefig(f'{img_dir}/hybrid_tsne.png', dpi=300, bbox_inches='tight')
    plt.show()


    # ======================================================================
    # ## Section 12: LIME explanation
    # ======================================================================

    from lime import lime_image
    from skimage.segmentation import mark_boundaries  # was missing an import in the original

    def lime_predict_fn(images):
        images_tensor = torch.tensor(images).permute(0, 3, 1, 2).float().cuda()
        with torch.no_grad():
            outputs = model_to_evaluate(images_tensor)
        return F.softmax(outputs, dim=1).cpu().numpy()

    lime_explainer = lime_image.LimeImageExplainer()

    lime_image_idx = 0
    lime_test_image = test_loader.dataset[lime_image_idx][0].unsqueeze(0)
    lime_test_image_np = lime_test_image.squeeze(0).cpu().numpy().transpose(1, 2, 0)  # CHW -> HWC

    lime_explanation = lime_explainer.explain_instance(
        lime_test_image_np,
        lime_predict_fn,
        top_labels=5,
        hide_color=0,
        num_samples=1000,
    )

    lime_label_to_explain = lime_explanation.top_labels[0]
    lime_temp, lime_mask = lime_explanation.get_image_and_mask(
        lime_label_to_explain, positive_only=True, num_features=10, hide_rest=False
    )

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(lime_test_image_np)
    plt.title(f"{CURRENT_DATASET} - Original Image")

    plt.subplot(1, 2, 2)
    plt.imshow(mark_boundaries(lime_temp, lime_mask))
    plt.title(f"{CURRENT_DATASET} - LIME Explanation for Class: {lime_label_to_explain}")
    plt.show()


    # ======================================================================
    # ## Section 13: SHAP explanation
    # ======================================================================

    # Replacement for the "## 13. SHAP explanation" cell.
    # Uses shap.GradientExplainer instead of shap.DeepExplainer.
    #
    # Why this avoids the earlier error: DeepExplainer installs backward hooks on
    # every layer and replays a modified backward pass, which is what collided
    # with ResNet's in-place "out += identity" residual add. GradientExplainer
    # instead computes attributions via expected gradients (interpolating between
    # background and input, averaging normal .backward() gradients) — no custom
    # hooks on internal layers, so it doesn't care whether ops are in-place.
    # No monkey-patching of torchvision.models.resnet needed.

    import shap

    shap_image_idx = 0
    shap_test_image, _ = test_loader.dataset[shap_image_idx]
    shap_test_image = shap_test_image.unsqueeze(0)
    print("Shape of the image tensor before conversion:", shap_test_image.shape)

    # Background sample for the explainer (small batch of real images).
    # FIX: hardcoded range(50) crashes if test_loader.dataset has fewer than 50 images
    # (e.g. QUICK_TEST_MODE caps it to 16). Cap the background size to whatever is
    # actually available.
    shap_n_background = min(50, len(test_loader.dataset))
    shap_background = torch.stack([test_loader.dataset[i][0] for i in range(shap_n_background)]).cuda()

    model_to_evaluate.eval()  # GradientExplainer expects eval mode; no ReLU/BasicBlock patching needed

    shap_explainer = shap.GradientExplainer(model_to_evaluate, shap_background)

    # GradientExplainer's shap_values signature differs slightly from DeepExplainer:
    # ranked_outputs=1 returns the top predicted class per image (plus its index),
    # which mirrors what the original DeepExplainer + top_labels-style flow gave us.
    shap_values, indexes = shap_explainer.shap_values(
        shap_test_image.cuda(), ranked_outputs=1
    )
    shap_values_class = shap_values[0]  # attributions for the top predicted class

    shap_map = np.mean(np.abs(shap_values_class), axis=0)
    shap_map = (shap_map - shap_map.min()) / (shap_map.max() - shap_map.min() + 1e-8)
    if shap_map.ndim == 3:
        shap_map = shap_map.mean(axis=0)

    shap_display_image = shap_test_image.squeeze(0).permute(1, 2, 0).cpu().numpy()
    shap_display_image = np.clip(shap_display_image, 0, 1)

    predicted_class_idx = int(indexes[0][0])
    predicted_class_name = (
        dataset.classes[predicted_class_idx] if "dataset" in globals() else predicted_class_idx
    )

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(shap_display_image)
    plt.axis('off')
    plt.title(f'{CURRENT_DATASET} - Original Image')

    plt.subplot(1, 2, 2)
    plt.imshow(shap_display_image)
    plt.imshow(shap_map, cmap='jet', alpha=0.5)
    plt.axis('off')
    plt.title(f'SHAP Explanation (class: {predicted_class_name})')
    plt.show()


    # ======================================================================
    # ## Section 13b: Table 3 / Table 5 / param breakdown
    # ======================================================================

    # --- Table 3 style metrics (Kather5k only) ---
    # Accuracy is pulled directly from comparison_results (Section 8 FINAL COMPARISON) --
    # same trained models, same test_loader -- so it can never mismatch that table.
    # NOTE: accuracy_score/precision_score/recall_score/f1_score are already imported at
    # notebook top-level (Cell 5) -- re-importing them here (as a local import inside
    # run_pipeline) used to make Python treat them as LOCAL to this whole function, which
    # broke the earlier plot_confusion_matrix() closure (Section 9) with a
    # "free variable ... not associated with a value" error, since that nested function
    # runs before this import line does. Removed; just use the top-level imports.
    # (`pandas as pd` was ALSO redundantly re-imported here -- removed for the same
    # reason: it's used earlier, at Section 9b's `table10 = pd.DataFrame(...)`, which
    # would otherwise hit the same UnboundLocalError once the sklearn.metrics one above
    # was fixed. Both are already imported at notebook top-level, Cell 5.)

    def compute_metrics_table(models_dict, data_loader, comparison_results):
        rows = {}
        for name, model in models_dict.items():
            preds, labels = get_predictions_and_labels(model, data_loader)
            rows[name] = {
                'Accuracy':  comparison_results[name]['Test Accuracy (%)'] / 100,  # from Section 8
                'Precision': precision_score(labels, preds, average='weighted', zero_division=0),
                'Recall':    recall_score(labels, preds, average='weighted', zero_division=0),
                'F1-score':  f1_score(labels, preds, average='weighted', zero_division=0),
            }
        df = pd.DataFrame(rows).T[['Accuracy', 'Precision', 'Recall', 'F1-score']]
        return df.round(4)

    models_dict = {
        'LoRaS-CT': loras_ct_model,
        'MHA-Net (baseline)': mha_net_model,
    }

    metrics_table = compute_metrics_table(models_dict, test_loader, comparison_results)
    print(f"Table 3: Performance metrics on {CURRENT_DATASET} (Accuracy matches Section 8 FINAL COMPARISON exactly)\n")
    print(metrics_table.to_string())

    # --- Table 5 style efficiency comparison (Kather5k only) ---
    # Pulls DIRECTLY from comparison_results (Section 8 "FINAL COMPARISON") instead of
    # re-measuring with a separate function/methodology. This is the fix for the earlier
    # mismatch (Table 5 used to call a differently-scoped, buggy measurement function that
    # produced different numbers -- and even an identical memory figure for both models --
    # from the ones already computed and printed in Section 8). Now there is exactly ONE
    # measurement of Params/FLOPs/Inference Time/Peak Memory in the whole notebook, and
    # every downstream table just displays it.
    efficiency_table = pd.DataFrame(comparison_results).T[
        ['Params (M)', 'FLOPs (G)', 'Inference Time (ms)', 'Peak Memory (MB)', 'Test Accuracy (%)']
    ].round(4)
    print(f"Table 5: Computational efficiency on {CURRENT_DATASET} (identical to Section 8 FINAL COMPARISON)\n")
    print(efficiency_table.to_string())

    # --- Param breakdown table: trainable vs transformer+head-only vs shared CNN backbone ---
    def count_all_parameters(module):
        return sum(p.numel() for p in module.parameters())

    def param_breakdown(models_dict):
        rows = {}
        backbone_params = None
        for name, model in models_dict.items():
            total_trainable = count_parameters(model)  # requires_grad only
            if hasattr(model, 'resnet') and hasattr(model, 'densenet'):
                head_only = count_custom_parameters(model, exclude_list=[model.resnet, model.densenet])
                if backbone_params is None:
                    backbone_params = count_all_parameters(model.resnet) + count_all_parameters(model.densenet)
            else:
                head_only = total_trainable
            rows[name] = {
                'Total trainable params': f'{total_trainable:,}',
                'Transformer+head only (excl. CNN backbones)': f'{head_only:,}',
            }
        df = pd.DataFrame(rows).T
        return df, backbone_params

    param_table, shared_backbone_params = param_breakdown(models_dict)
    print(param_table.to_string())
    print(f"\nShared CNN backbone params (ResNet18+DenseNet121 features, same for both models): {shared_backbone_params:,}")

    # --- Consistency check against Section 8 FINAL COMPARISON ---
    print(f"\n[{CURRENT_DATASET}] Consistency check vs. Section 8 FINAL COMPARISON (Params (M)):")
    for name in models_dict:
        total_trainable = count_parameters(models_dict[name])
        reported = comparison_results[name]['Params (M)'] * 1e6
        status = "OK" if abs(total_trainable - reported) < 1 else "MISMATCH -- investigate"
        print(f"  {name}: param_breakdown={total_trainable:,}  vs  comparison_results={reported:,.0f}  [{status}]")


    # ======================================================================
    # ## Section 14: Statistical significance analysis (multi-seed + paired t-test)
    # ======================================================================

    model_builders = {
        'LoRaS-CT': lambda: HybridStudentModel(num_classes, grid_size=3).cuda(),
        'MHA-Net (baseline)': lambda: MHANetBaseline(num_classes, grid_size=3).cuda(),
    }

    multiseed_results = {name: {'accuracy': [], 'f1': [], 'precision': [], 'recall': []} for name in model_builders}

    for seed in SEED_LIST:
        print(f"\n{'#'*70}\n# SEED {seed}\n{'#'*70}")
        set_all_seeds(seed)
        seed_train_loader, seed_val_loader, seed_test_loader = make_split(dataset_path, seed)

        for name, builder in model_builders.items():
            set_all_seeds(seed)  # re-seed right before init so weight init is also seed-controlled
            model = builder()
            criterion = DistillationLoss(alpha=0.5, temperature=3.0)
            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

            model, *_ = train_model_with_distillation(
                model, teacher_models, seed_train_loader, seed_val_loader,
                criterion, optimizer, num_epochs=STAT_NUM_EPOCHS
            )

            acc, _, y_true, y_pred = evaluate_model(model, seed_test_loader, criterion.ce_loss, return_predictions=True)
            f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
            prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
            rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)

            multiseed_results[name]['accuracy'].append(acc)
            multiseed_results[name]['f1'].append(f1)
            multiseed_results[name]['precision'].append(prec)
            multiseed_results[name]['recall'].append(rec)

            print(f"[seed {seed}] {name}: Acc={acc:.2f}%  F1={f1:.4f}")

            del model
            torch.cuda.empty_cache()
            gc.collect()

    # --- Mean +/- SD summary table ---
    summary_rows = {}
    for name, m in multiseed_results.items():
        summary_rows[name] = {
            'Accuracy (%)': f"{np.mean(m['accuracy']):.2f} +/- {np.std(m['accuracy']):.2f}",
            'F1-score':     f"{np.mean(m['f1']):.4f} +/- {np.std(m['f1']):.4f}",
            'Precision':    f"{np.mean(m['precision']):.4f} +/- {np.std(m['precision']):.4f}",
            'Recall':       f"{np.mean(m['recall']):.4f} +/- {np.std(m['recall']):.4f}",
            'N runs':       len(m['accuracy']),
        }
    stat_summary_df = pd.DataFrame(summary_rows).T
    print(f"Table: Mean +/- SD over {N_SEEDS} independent seeds -- {CURRENT_DATASET}\n")
    print(stat_summary_df.to_string())

    # --- Paired t-test: LoRaS-CT vs MHA-Net accuracy, matched by seed ---
    # FIX: a paired t-test needs at least 2 paired observations to estimate variance;
    # with N_SEEDS=1 (e.g. QUICK_TEST_MODE) scipy divides by zero and returns nan/nan
    # with a RuntimeWarning instead of a crash -- but it's a meaningless "result", not
    # an error, so we now skip the test explicitly rather than print a nan p-value.
    acc_a = multiseed_results['LoRaS-CT']['accuracy']
    acc_b = multiseed_results['MHA-Net (baseline)']['accuracy']
    if N_SEEDS < 2:
        print(f"\nPaired t-test skipped: N_SEEDS={N_SEEDS} (need >= 2 seeds to estimate variance).")
        print("  This is expected under QUICK_TEST_MODE. Set QUICK_TEST_MODE = False for a real test.")
    else:
        t_stat, p_value = stats.ttest_rel(acc_a, acc_b)
        print(f"\nPaired t-test (LoRaS-CT vs MHA-Net, accuracy, N={N_SEEDS} seeds):")
        print(f"  t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")
        print(f"  {'Statistically significant at alpha=0.05' if p_value < 0.05 else 'NOT statistically significant at alpha=0.05'}")
        print("  NOTE: with N_SEEDS=3 this test has very low statistical power -- treat the p-value")
        print("  as indicative only. Increase N_SEEDS to 5-10 for the camera-ready submission.")

    # --- Plot: bar chart with error bars ---
    fig, ax = plt.subplots(figsize=(6, 4))
    names = list(multiseed_results.keys())
    means = [np.mean(multiseed_results[n]['accuracy']) for n in names]
    stds = [np.std(multiseed_results[n]['accuracy']) for n in names]
    colors = ['#2E86AB', '#A23B72']
    ax.bar(names, means, yerr=stds, capsize=8, color=colors[:len(names)])
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: Test Accuracy Mean +/- SD over {N_SEEDS} Seeds')
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.text(i, m + s + 0.3, f'{m:.2f}+/-{s:.2f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{img_dir}/stat_significance_barplot.png', dpi=150)
    plt.show()


    # ======================================================================
    # ## Section 16: Grad-CAM explainability
    # ======================================================================

    # Target the last conv block of the ResNet-18 branch (before it's concatenated with
    # DenseNet and fed into the transformer) -- this is where "which pixels drove the
    # decision" is most interpretable for a pathologist.
    gradcam_target_layer = model_to_evaluate.resnet.features[-1]
    gradcam = GradCAM(model_to_evaluate, gradcam_target_layer)

    n_examples = 4
    fig, axes = plt.subplots(2, n_examples, figsize=(4 * n_examples, 8))

    for i in range(n_examples):
        img_tensor, true_label = test_loader.dataset[i]
        input_tensor = img_tensor.unsqueeze(0).cuda()

        cam, pred_class = gradcam.generate(input_tensor)

        img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

        true_name = dataset.classes[true_label] if "dataset" in globals() else true_label
        pred_name = dataset.classes[pred_class] if "dataset" in globals() else pred_class

        axes[0, i].imshow(img_np)
        axes[0, i].axis('off')
        axes[0, i].set_title(f"True: {true_name}", fontsize=9)

        axes[1, i].imshow(img_np)
        axes[1, i].imshow(cam, cmap='jet', alpha=0.5)
        axes[1, i].axis('off')
        axes[1, i].set_title(f"Grad-CAM (pred: {pred_name})", fontsize=9)

    plt.suptitle(f"Grad-CAM explanations -- {CURRENT_DATASET} ({model_to_evaluate_name})", y=1.02)
    plt.tight_layout()
    plt.savefig(f'{img_dir}/gradcam_examples.png', dpi=150, bbox_inches='tight')
    plt.show()

    gradcam.remove()


    # ======================================================================
    # ## Section 17: Component-wise ablation study
    # ======================================================================

    ABLATION_EPOCHS = GLOBAL_NUM_EPOCHS  # bump up for camera-ready

    def build_full_model():
        return HybridStudentModel(num_classes, grid_size=3).cuda()

    def build_no_lowrank():
        return GenericHybridModel(
            num_classes, attn_factory=lambda: FullRankSparseAttention(768, GLOBAL_NUM_HEADS, sparsity_ratio=0.5), grid_size=3
        ).cuda()

    def build_no_sparsity():
        return GenericHybridModel(
            num_classes, attn_factory=lambda: LowRankSparseMultiheadAttention(768, GLOBAL_NUM_HEADS, rank=GLOBAL_RANK, sparsity_ratio=1.0),
            grid_size=3
        ).cuda()

    def build_resnet_only():
        return SingleBackboneHybridModel(num_classes, backbone='resnet', grid_size=3).cuda()

    def build_densenet_only():
        return SingleBackboneHybridModel(num_classes, backbone='densenet', grid_size=3).cuda()

    ablation_configs = {
        'Full LoRaS-CT (all components)':    (build_full_model,    True),
        'w/o Low-Rank Factorization':        (build_no_lowrank,    True),
        'w/o Sparsity (top-k masking)':      (build_no_sparsity,   True),
        'w/o Knowledge Distillation':        (build_full_model,    False),
        'w/o DenseNet branch (ResNet only)': (build_resnet_only,   True),
        'w/o ResNet branch (DenseNet only)': (build_densenet_only, True),
    }

    ablation_results = {}
    for name, (builder, use_kd) in ablation_configs.items():
        print(f"\n{'='*60}\nAblation: {name}\n{'='*60}")
        set_all_seeds(42)
        model = builder()

        if use_kd:
            criterion = DistillationLoss(alpha=0.5, temperature=3.0)
            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
            model, *_ = train_model_with_distillation(
                model, teacher_models, train_loader, val_loader, criterion, optimizer, num_epochs=ABLATION_EPOCHS
            )
            eval_criterion = criterion.ce_loss
        else:
            eval_criterion = nn.CrossEntropyLoss()
            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
            train_model_plain(model, train_loader, val_loader, eval_criterion, optimizer, num_epochs=ABLATION_EPOCHS)

        acc, _, y_true, y_pred = evaluate_model(model, test_loader, eval_criterion, return_predictions=True)
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        params = count_parameters(model)

        ablation_results[name] = {'Accuracy (%)': acc, 'F1-score': f1, 'Params (M)': params / 1e6}
        print(f"[{name}] Acc={acc:.2f}%  F1={f1:.4f}  Params={params/1e6:.2f}M")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    ablation_df = pd.DataFrame(ablation_results).T[['Accuracy (%)', 'F1-score', 'Params (M)']]
    full_acc = ablation_df.loc['Full LoRaS-CT (all components)', 'Accuracy (%)']
    ablation_df['Delta Accuracy (pp)'] = (ablation_df['Accuracy (%)'] - full_acc).round(2)
    print(f"Table: Component-wise ablation study -- {CURRENT_DATASET}\n")
    print(ablation_df.round(4).to_string())

    fig, ax = plt.subplots(figsize=(9, 5))
    names = list(ablation_results.keys())
    accs = [ablation_results[n]['Accuracy (%)'] for n in names]
    colors = ['#2E86AB'] + ['#A23B72'] * (len(names) - 1)
    bars = ax.barh(names, accs, color=colors)
    ax.axvline(full_acc, color='gray', linestyle='--', label='Full model accuracy')
    ax.set_xlabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: Component-wise Ablation Study')
    ax.legend()
    for bar, acc in zip(bars, accs):
        ax.text(acc + 0.3, bar.get_y() + bar.get_height() / 2, f'{acc:.2f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{img_dir}/ablation_study.png', dpi=150)
    plt.show()


    # ======================================================================
    # ## Section 18: Comparison with pathology foundation models
    # ======================================================================

    foundation_models_to_test = [] if QUICK_TEST_MODE else ['UNI', 'CONCH', 'Virchow']  # add 'HIPT' once you have local weights
    if QUICK_TEST_MODE:
        print("QUICK_TEST_MODE: skipping foundation-model downloads (multi-GB gated weights).")
        print("Set QUICK_TEST_MODE = False in Cell 5 to run this section for real.")
    elif not globals().get('hf_login_ok', False):
        print("Hugging Face login was not successful (see the cell above) -- skipping all three")
        print("gated foundation models rather than hitting the same 401 three times.")
        foundation_models_to_test = []
    foundation_results = {}

    for fm_name in foundation_models_to_test:
        print(f"\n{'='*60}\nFoundation model: {fm_name}\n{'='*60}")
        try:
            encoder, embed_dim = load_foundation_encoder(fm_name)
            model = LinearProbeHead(encoder, embed_dim, num_classes).cuda()

            optimizer = optim.Adam(model.head.parameters(), lr=1e-3)  # only the head is trained
            criterion = nn.CrossEntropyLoss()
            train_model_plain(model, train_loader, val_loader, criterion, optimizer,
                               num_epochs=FOUNDATION_MODEL_EPOCHS)

            acc, _, y_true, y_pred = evaluate_model(model, test_loader, criterion, return_predictions=True)
            f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
            params = count_parameters(model)  # trainable only == the head, by design

            dummy_input = torch.randn(1, 3, 224, 224).cuda()
            inf_time = measure_inference_time(model, dummy_input)
            memory = measure_memory(model, dummy_input)

            foundation_results[fm_name] = {
                'Test Accuracy (%)': acc, 'F1-score': f1,
                'Trainable Params (M)': params / 1e6,
                'Total Params (M)': sum(p.numel() for p in model.parameters()) / 1e6,
                'Inference Time (ms)': inf_time, 'Peak Memory (MB)': memory,
            }
            print(f"[{fm_name}] Acc={acc:.2f}%  F1={f1:.4f}")

            del model, encoder
            torch.cuda.empty_cache()
            gc.collect()

        except Exception as e:
            print(f"[{fm_name}] SKIPPED -- {type(e).__name__}: {e}")
            print("  (Needs internet + HF gated-license access in this environment. Run "
                  "`huggingface-cli login` after accepting the license on the model's HF page.)")
            foundation_results[fm_name] = {
                'Test Accuracy (%)': np.nan, 'F1-score': np.nan,
                'Trainable Params (M)': np.nan, 'Total Params (M)': np.nan,
                'Inference Time (ms)': np.nan, 'Peak Memory (MB)': np.nan,
            }

    combined_results = dict(foundation_results)
    combined_results['LoRaS-CT (ours)'] = {
        'Test Accuracy (%)': ablation_results['Full LoRaS-CT (all components)']['Accuracy (%)']
            if 'ablation_results' in globals() else np.nan,
        'F1-score': ablation_results['Full LoRaS-CT (all components)']['F1-score']
            if 'ablation_results' in globals() else np.nan,
        'Trainable Params (M)': count_parameters(loras_ct_model) / 1e6,
        'Total Params (M)': sum(p.numel() for p in loras_ct_model.parameters()) / 1e6,
        'Inference Time (ms)': np.nan, 'Peak Memory (MB)': np.nan,
    }

    foundation_comparison_df = pd.DataFrame(combined_results).T
    print(f"Table: LoRaS-CT vs pathology foundation models (linear probe) -- {CURRENT_DATASET}\n")
    print(foundation_comparison_df.round(4).to_string())

    fig, ax = plt.subplots(figsize=(8, 5))
    valid = foundation_comparison_df.dropna(subset=['Test Accuracy (%)'])
    bar_colors = ['#F18F01'] * (len(valid) - 1) + ['#2E86AB'] if 'LoRaS-CT (ours)' in valid.index else ['#F18F01'] * len(valid)
    ax.bar(valid.index, valid['Test Accuracy (%)'], color=bar_colors)
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'LoRaS-CT vs Pathology Foundation Models (linear probe, {CURRENT_DATASET})')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig(f'{img_dir}/foundation_model_comparison.png', dpi=150)
    plt.show()


    # ======================================================================
    # ## Section 19: K-Fold cross-validation + confidence intervals
    # ======================================================================

    # --- K-Fold Cross-Validation with 95% Confidence Intervals ---
    from sklearn.model_selection import KFold, GroupKFold
    from scipy import stats as scipy_stats

    K_FOLDS = 2 if QUICK_TEST_MODE else 5
    KFOLD_NUM_EPOCHS = GLOBAL_NUM_EPOCHS  # bump up for camera-ready

    full_ds_for_kfold = datasets.ImageFolder(dataset_path)
    all_indices = np.arange(len(full_ds_for_kfold))
    kfold_groups = get_patient_groups(full_ds_for_kfold)

    if kfold_groups is not None:
        print(f"[{CURRENT_DATASET}] Patient/slide IDs detected -- using GroupKFold "
              f"(no patient split across folds).")
        kf = GroupKFold(n_splits=K_FOLDS)
        fold_iter = kf.split(all_indices, groups=kfold_groups)
    else:
        kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
        fold_iter = kf.split(all_indices)

    # reuse the same model_builders dict defined in Section 14
    kfold_results = {name: [] for name in model_builders}

    for fold_idx, (trainval_idx, test_idx) in enumerate(fold_iter):
        print(f"\n{'#'*70}\n# FOLD {fold_idx + 1}/{K_FOLDS}\n{'#'*70}")
        set_all_seeds(42 + fold_idx)

        if kfold_groups is not None:
            trainval_groups = kfold_groups[trainval_idx]
            gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42 + fold_idx)
            tr_sub, val_sub = next(gss.split(trainval_idx, groups=trainval_groups))
            train_idx, val_idx = trainval_idx[tr_sub], trainval_idx[val_sub]
        else:
            n_val = int(len(trainval_idx) * 0.1)
            val_idx = trainval_idx[:n_val]
            train_idx = trainval_idx[n_val:]

        train_subset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
        val_subset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
        test_subset = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)
        train_subset = quick_subset(train_subset, QUICK_TEST_MAX_TRAIN)
        val_subset = quick_subset(val_subset, QUICK_TEST_MAX_VAL)
        test_subset = quick_subset(test_subset, QUICK_TEST_MAX_TEST)

        fold_train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
        fold_val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
        fold_test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=2)

        for name, builder in model_builders.items():
            set_all_seeds(42 + fold_idx)
            model = builder()
            criterion = DistillationLoss(alpha=0.5, temperature=3.0)
            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
            model, *_ = train_model_with_distillation(
                model, teacher_models, fold_train_loader, fold_val_loader, criterion, optimizer,
                num_epochs=KFOLD_NUM_EPOCHS
            )
            acc, _ = evaluate_model(model, fold_test_loader, criterion.ce_loss)
            kfold_results[name].append(acc)
            print(f"[Fold {fold_idx + 1}] {name}: Test Acc = {acc:.2f}%")

            del model
            torch.cuda.empty_cache()
            gc.collect()

    # --- Summary table (mean, std, 95% CI) + paired t-test across folds ---
    def confidence_interval_95(values):
        values = np.array(values)
        mean = values.mean()
        if len(values) > 1:
            sem = scipy_stats.sem(values)
            ci = scipy_stats.t.interval(0.95, df=len(values) - 1, loc=mean, scale=sem)
        else:
            ci = (mean, mean)
        return mean, values.std(), ci

    kfold_summary_rows = {}
    for name, accs in kfold_results.items():
        mean, std, ci = confidence_interval_95(accs)
        kfold_summary_rows[name] = {
            'Mean Accuracy (%)': round(mean, 2),
            'Std Dev': round(std, 2),
            '95% CI Lower': round(ci[0], 2),
            '95% CI Upper': round(ci[1], 2),
            'K': len(accs),
        }
    kfold_summary_df = pd.DataFrame(kfold_summary_rows).T
    print(f"Table: {K_FOLDS}-Fold Cross-Validation Results with 95% Confidence Intervals -- {CURRENT_DATASET}\n")
    print(kfold_summary_df.to_string())

    # Same guard as Section 14's t-test -- see comment there.
    if K_FOLDS < 2:
        print(f"\nPaired t-test skipped: K_FOLDS={K_FOLDS} (need >= 2 folds to estimate variance).")
    else:
        t_stat, p_value = scipy_stats.ttest_rel(kfold_results['LoRaS-CT'], kfold_results['MHA-Net (baseline)'])
        print(f"\nPaired t-test across {K_FOLDS} folds (LoRaS-CT vs MHA-Net): t={t_stat:.4f}, p={p_value:.4f}")
        print(f"  {'Statistically significant at alpha=0.05' if p_value < 0.05 else 'NOT statistically significant at alpha=0.05'}")

    fig, ax = plt.subplots(figsize=(6, 4))
    names = list(kfold_results.keys())
    means = [np.mean(kfold_results[n]) for n in names]
    cis = [confidence_interval_95(kfold_results[n])[2] for n in names]
    errs = np.array([[m - ci[0], ci[1] - m] for m, ci in zip(means, cis)]).T
    ax.bar(names, means, yerr=errs, capsize=8, color=['#2E86AB', '#A23B72'])
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: {K_FOLDS}-Fold CV Mean Accuracy with 95% CI')
    plt.tight_layout()
    plt.savefig(f'{img_dir}/kfold_cv_results.png', dpi=150)
    plt.show()


    # ======================================================================
    # ## Section 20: Hyperparameter ablation -- layers & heads
    # ======================================================================

    HPARAM_ABLATION_EPOCHS = GLOBAL_NUM_EPOCHS  # bump up for camera-ready

    layer_configs = [1, 2] if QUICK_TEST_MODE else [1, 2, 3, 4]
    head_configs = [4, 8] if QUICK_TEST_MODE else [2, 4, 8, 16]  # all divide embed_dim=768 evenly

    hparam_results_layers = {}
    for nl in layer_configs:
        print(f"\n--- num_layers = {nl} ---")
        set_all_seeds(42)
        model = HybridStudentModel(num_classes, num_layers=nl, grid_size=3).cuda()
        criterion = DistillationLoss(alpha=0.5, temperature=3.0)
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
        model, *_ = train_model_with_distillation(
            model, teacher_models, train_loader, val_loader, criterion, optimizer,
            num_epochs=HPARAM_ABLATION_EPOCHS
        )
        acc, _ = evaluate_model(model, test_loader, criterion.ce_loss)
        params = count_parameters(model)
        hparam_results_layers[nl] = {'Test Accuracy (%)': acc, 'Params (M)': params / 1e6}
        print(f"[num_layers={nl}] Acc={acc:.2f}%  Params={params/1e6:.2f}M")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    hparam_results_heads = {}
    for nh in head_configs:
        print(f"\n--- num_heads = {nh} ---")
        set_all_seeds(42)
        model = HybridStudentModel(num_classes, num_heads=nh, grid_size=3).cuda()
        criterion = DistillationLoss(alpha=0.5, temperature=3.0)
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
        model, *_ = train_model_with_distillation(
            model, teacher_models, train_loader, val_loader, criterion, optimizer,
            num_epochs=HPARAM_ABLATION_EPOCHS
        )
        acc, _ = evaluate_model(model, test_loader, criterion.ce_loss)
        hparam_results_heads[nh] = {'Test Accuracy (%)': acc}
        print(f"[num_heads={nh}] Acc={acc:.2f}%")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # --- Tables + plots ---
    layers_df = pd.DataFrame(hparam_results_layers).T
    layers_df.index.name = 'num_layers'
    print(f"Table: Ablation over number of transformer layers -- {CURRENT_DATASET}\n")
    print(layers_df.round(4).to_string())

    heads_df = pd.DataFrame(hparam_results_heads).T
    heads_df.index.name = 'num_heads'
    print(f"\nTable: Ablation over number of attention heads -- {CURRENT_DATASET}\n")
    print(heads_df.round(4).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    axes[0].plot(list(hparam_results_layers.keys()),
                 [v['Test Accuracy (%)'] for v in hparam_results_layers.values()],
                 marker='o', color='#2E86AB')
    axes[0].set_xlabel('Number of Transformer Layers')
    axes[0].set_ylabel('Test Accuracy (%)')
    axes[0].set_title(f'{CURRENT_DATASET}: Accuracy vs. Number of Layers')
    axes[0].set_xticks(layer_configs)

    axes[1].plot(list(hparam_results_heads.keys()),
                 [v['Test Accuracy (%)'] for v in hparam_results_heads.values()],
                 marker='s', color='#A23B72')
    axes[1].set_xlabel('Number of Attention Heads')
    axes[1].set_ylabel('Test Accuracy (%)')
    axes[1].set_title(f'{CURRENT_DATASET}: Accuracy vs. Number of Attention Heads')
    axes[1].set_xticks(head_configs)

    plt.tight_layout()
    plt.savefig(f'{img_dir}/hparam_ablation_layers_heads.png', dpi=150)
    plt.show()

    print("\nNote: the KD on/off ablation for this same reviewer comment is already reported")
    print("in Section 17 ('w/o Knowledge Distillation' row of the component-wise ablation table).")


    # ======================================================================
    # ## Section 21: Teacher standalone performance + weighted vs simple averaging
    # ======================================================================

    # --- Teacher standalone performance, BEFORE fine-tuning ---
    TEACHER_FT_EPOCHS = GLOBAL_NUM_EPOCHS  # bump up for camera-ready
    teacher_names = ['ViT-B/16', 'DeiT-B/16', 'Swin-B/4']

    def evaluate_teacher_standalone(teacher, data_loader):
        acc, _ = evaluate_model(teacher, data_loader, nn.CrossEntropyLoss())
        return acc

    print(f"[{CURRENT_DATASET}] Teacher accuracy BEFORE fine-tuning")
    print("(ImageNet-pretrained backbone, but the num_classes-sized classification head")
    print(" has never been trained on this dataset -- expect close to chance level):\n")
    pre_ft_accs = {}
    for name, teacher in zip(teacher_names, teacher_models):
        acc = evaluate_teacher_standalone(teacher, test_loader)
        pre_ft_accs[name] = acc
        print(f"  {name}: {acc:.2f}%")

    # --- Fine-tune each teacher on the target dataset, then re-measure ---
    # NOTE: this mutates teacher_models in place. If you re-run this cell, the teachers
    # just continue training further (idempotent-ish, not a hard reset). Re-run Cell 9
    # (teacher_vit/teacher_deit/teacher_swin construction) first if you want a clean restart.
    print(f"[{CURRENT_DATASET}] Fine-tuning each teacher on the target dataset...\n")
    for name, teacher in zip(teacher_names, teacher_models):
        print(f"--- {name} ---")
        optimizer = optim.SGD(teacher.parameters(), lr=1e-4, momentum=0.9)  # low LR full fine-tune
        criterion = nn.CrossEntropyLoss()
        train_model_plain(teacher, train_loader, val_loader, criterion, optimizer, num_epochs=TEACHER_FT_EPOCHS)

    post_ft_accs = {}
    print("\nTeacher accuracy AFTER fine-tuning:\n")
    for name, teacher in zip(teacher_names, teacher_models):
        acc = evaluate_teacher_standalone(teacher, test_loader)
        post_ft_accs[name] = acc
        print(f"  {name}: {acc:.2f}%")

    teacher_comparison_df = pd.DataFrame({
        'Before fine-tuning (%)': pre_ft_accs,
        'After fine-tuning (%)': post_ft_accs,
    })
    print(f"\nTable: Teacher standalone performance, before vs. after fine-tuning -- {CURRENT_DATASET}\n")
    print(teacher_comparison_df.round(2).to_string())

    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = np.arange(len(teacher_names))
    width = 0.35
    ax.bar(x - width/2, list(pre_ft_accs.values()), width, label='Before fine-tuning', color='#F18F01')
    ax.bar(x + width/2, list(post_ft_accs.values()), width, label='After fine-tuning', color='#2E86AB')
    ax.set_xticks(x)
    ax.set_xticklabels(teacher_names)
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: Teacher Standalone Accuracy Before vs. After Fine-tuning')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{img_dir}/teacher_finetuning_comparison.png', dpi=150)
    plt.show()

    # --- Simple averaging vs. learned weighted averaging in MTKD ---
    class WeightedDistillationLoss(nn.Module):
        """Same as DistillationLoss, but combines multiple teacher logits with a LEARNED
        softmax-normalized weight per teacher (jointly optimized with the student) instead
        of the fixed simple average used in the original DistillationLoss."""
        def __init__(self, num_teachers, alpha=0.5, temperature=3.0):
            super().__init__()
            self.alpha = alpha
            self.temperature = temperature
            self.ce_loss = nn.CrossEntropyLoss()
            self.kl_div = nn.KLDivLoss(reduction="batchmean")
            self.teacher_weights = nn.Parameter(torch.ones(num_teachers))

        def combine(self, teacher_logits_list):
            w = F.softmax(self.teacher_weights, dim=0)
            return sum(wi * tl for wi, tl in zip(w, teacher_logits_list))

        def forward(self, student_logits, teacher_logits_list, ground_truth):
            combined_teacher_logits = self.combine(teacher_logits_list)
            hard_loss = self.ce_loss(student_logits, ground_truth)
            soft_loss = self.kl_div(
                F.log_softmax(student_logits / self.temperature, dim=1),
                F.softmax(combined_teacher_logits / self.temperature, dim=1)
            ) * (self.temperature ** 2)
            return self.alpha * soft_loss + (1 - self.alpha) * hard_loss


    def train_model_with_weighted_distillation(student_model, teacher_models, train_loader, val_loader,
                                                distillation_criterion, optimizer, num_epochs=1):
        """Same loop as train_model_with_distillation, but keeps teacher logits as a LIST
        (not pre-averaged) so WeightedDistillationLoss can combine them with learned weights."""
        for epoch in range(num_epochs):
            student_model.train()
            for images, labels in train_loader:
                images, labels = images.cuda(), labels.cuda()
                optimizer.zero_grad()
                student_outputs = student_model(images)
                with torch.no_grad():
                    teacher_logits_list = [teacher(images) for teacher in teacher_models]
                loss = distillation_criterion(student_outputs, teacher_logits_list, labels)
                loss.backward()
                optimizer.step()
            train_acc, _ = evaluate_model(student_model, train_loader, distillation_criterion.ce_loss)
            val_acc, val_loss = evaluate_model(student_model, val_loader, distillation_criterion.ce_loss)
            print(f'Epoch [{epoch+1}/{num_epochs}] Train Acc: {train_acc:.2f}%  Val Acc: {val_acc:.2f}%')
        return student_model


    KD_WEIGHTING_EPOCHS = GLOBAL_NUM_EPOCHS  # bump up for camera-ready

    set_all_seeds(42)
    model_simple = HybridStudentModel(num_classes, grid_size=3).cuda()
    criterion_simple = DistillationLoss(alpha=0.5, temperature=3.0)
    optimizer_simple = optim.SGD(model_simple.parameters(), lr=0.01, momentum=0.9)
    model_simple, *_ = train_model_with_distillation(
        model_simple, teacher_models, train_loader, val_loader, criterion_simple, optimizer_simple,
        num_epochs=KD_WEIGHTING_EPOCHS
    )
    acc_simple, _ = evaluate_model(model_simple, test_loader, criterion_simple.ce_loss)

    # free model_simple BEFORE building model_weighted, so the two students never sit in
    # GPU memory at the same time (they did briefly in an earlier version of this cell)
    del model_simple
    torch.cuda.empty_cache()
    gc.collect()

    set_all_seeds(42)
    model_weighted = HybridStudentModel(num_classes, grid_size=3).cuda()
    criterion_weighted = WeightedDistillationLoss(num_teachers=len(teacher_models), alpha=0.5, temperature=3.0).cuda()
    optimizer_weighted = optim.SGD(
        list(model_weighted.parameters()) + list(criterion_weighted.parameters()), lr=0.01, momentum=0.9
    )
    model_weighted = train_model_with_weighted_distillation(
        model_weighted, teacher_models, train_loader, val_loader, criterion_weighted, optimizer_weighted,
        num_epochs=KD_WEIGHTING_EPOCHS
    )
    acc_weighted, _ = evaluate_model(model_weighted, test_loader, criterion_weighted.ce_loss)

    learned_weights = F.softmax(criterion_weighted.teacher_weights.detach(), dim=0).cpu().numpy()

    kd_weighting_df = pd.DataFrame({
        'Simple averaging': {'Test Accuracy (%)': acc_simple},
        'Learned weighted averaging': {'Test Accuracy (%)': acc_weighted},
    }).T
    print(f"Table: Simple vs. learned-weighted teacher averaging in MTKD -- {CURRENT_DATASET}\n")
    print(kd_weighting_df.round(2).to_string())
    print(f"\nLearned teacher weights (softmax-normalized): "
          f"ViT={learned_weights[0]:.3f}  DeiT={learned_weights[1]:.3f}  Swin={learned_weights[2]:.3f}")

    del model_weighted
    torch.cuda.empty_cache()
    gc.collect()

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(['Simple avg', 'Learned weighted'], [acc_simple, acc_weighted], color=['#2E86AB', '#A23B72'])
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: MTKD Simple vs. Weighted Teacher Averaging')
    plt.tight_layout()
    plt.savefig(f'{img_dir}/kd_weighting_comparison.png', dpi=150)
    plt.show()


    # ======================================================================
    # ## Section 22: Patient-level split verification
    # ======================================================================

    import re

    def try_extract_patient_id(filename):
        """Attempts several common histopathology dataset patient/case-ID filename
        conventions. Returns None if no pattern matches -- the audit below treats that
        as 'cannot verify from filenames alone' rather than silently assuming safety."""
        patterns = [
            r'SOB_[A-Z]_[A-Z]+-(\d+-\d+)',   # BreakHis: SOB_B_TA-14-4659-40-001.png -> "14-4659"
            r'^(P\d+)',                        # generic P<digits>_...
            r'patient[_-]?(\d+)',              # patient_12_... / patient-12-...
            r'case[_-]?(\d+)',                 # case_12_...
        ]
        for pat in patterns:
            m = re.search(pat, filename, flags=re.IGNORECASE)
            if m:
                return m.group(1)
        return None


    def resolve_nested_subset_indices(subset):
        """random_split (Cell 7) was applied twice, so train_dataset/val_dataset are
        Subsets of a Subset. Walks back through .dataset/.indices to get indices into
        the ORIGINAL base ImageFolder (`dataset`)."""
        indices = list(subset.indices)
        base = subset.dataset
        while isinstance(base, torch.utils.data.Subset):
            indices = [base.indices[i] for i in indices]
            base = base.dataset
        return indices, base


    def audit_split_for_leakage(train_subset, val_subset, test_subset):
        train_idx, base_ds = resolve_nested_subset_indices(train_subset)
        val_idx, _ = resolve_nested_subset_indices(val_subset)
        test_idx, _ = resolve_nested_subset_indices(test_subset)

        filepaths = [s[0] for s in base_ds.samples]
        patient_ids = [try_extract_patient_id(os.path.basename(fp)) for fp in filepaths]
        n_matched = sum(1 for p in patient_ids if p is not None)

        print(f"Filenames with a recognizable patient/case ID pattern: {n_matched}/{len(filepaths)}")

        if n_matched == 0:
            print(f"\nNo patient/case ID could be parsed from filenames in the {CURRENT_DATASET} dataset folder.")
            print(f"For {CURRENT_DATASET}, this means no patient/slide metadata was recognized in the")
            print("filenames (or this dataset genuinely has no patient/slide grouping, e.g. a tile-level")
            print("texture benchmark like Kather5k/NCT100k). If that's expected for this dataset, state it")
            print("explicitly in the rebuttal rather than claiming a patient-level split it doesn't support.")
            print(f"If {CURRENT_DATASET} DOES have multiple patches per patient/slide (e.g. BreakHis),")
            print("extend try_extract_patient_id()'s regex patterns to match its actual filename")
            print("convention, then rerun this cell and verify zero overlap before reporting final numbers.")
            return None

        train_patients = set(patient_ids[i] for i in train_idx if patient_ids[i] is not None)
        val_patients = set(patient_ids[i] for i in val_idx if patient_ids[i] is not None)
        test_patients = set(patient_ids[i] for i in test_idx if patient_ids[i] is not None)

        overlap_train_test = train_patients & test_patients
        overlap_train_val = train_patients & val_patients
        overlap_val_test = val_patients & test_patients

        print(f"\nUnique patients -- train: {len(train_patients)}, val: {len(val_patients)}, test: {len(test_patients)}")
        print(f"Patient overlap train<->test: {len(overlap_train_test)}")
        print(f"Patient overlap train<->val:  {len(overlap_train_val)}")
        print(f"Patient overlap val<->test:   {len(overlap_val_test)}")

        if overlap_train_test or overlap_train_val or overlap_val_test:
            print("\nLEAKAGE DETECTED: at least one patient appears in more than one split.")
            print("ACTION NEEDED: re-split at the patient level (e.g. sklearn GroupShuffleSplit /")
            print("GroupKFold using patient_ids as the group key) before reporting final numbers.")
        else:
            print("\nNo patient overlap detected across splits (based on parsed IDs).")

        return {'train': train_patients, 'val': val_patients, 'test': test_patients}


    print(f"[{CURRENT_DATASET}] Auditing the split from Section 1 / Cell 7 (dataset_path = {dataset_path})\n")
    patient_audit_result = audit_split_for_leakage(train_dataset, val_dataset, test_dataset)


    # ============================================================
    # Collect this dataset's key results/tables to return to the driver loop
    # ============================================================
    dataset_results = {
        'comparison_results': comparison_results,
        'metrics_table_3': metrics_table,
        'efficiency_table_5': efficiency_table,
        'ablation_df': ablation_df,
        'stat_summary_df': stat_summary_df,
        'foundation_comparison_df': foundation_comparison_df,
        'kfold_summary_df': kfold_summary_df,
        'hparam_layers_df': layers_df,
        'hparam_heads_df': heads_df,
        'teacher_comparison_df': teacher_comparison_df,
        'kd_weighting_df': kd_weighting_df,
        'sota_comparison_df': sota_comparison_df,
        'patient_audit_result': patient_audit_result,
    }
    print(f"\n{'#'*80}\n# FINISHED PIPELINE FOR DATASET: {CURRENT_DATASET}\n{'#'*80}\n")
    return dataset_results


## 5. Driver — run the pipeline for the selected dataset(s)

Runs once per entry in `DATASETS_TO_RUN` (set in Section 2). Results for every dataset run are collected into `all_dataset_results`, keyed by dataset name.

In [ ]:
all_dataset_results = {}

for _ds_name in DATASETS_TO_RUN:
    _ds_cfg = DATASET_CONFIGS[_ds_name]

    # Pre-flight: skip cleanly (with a clear message) if the configured path(s) for this
    # dataset don't exist on disk yet, instead of crashing deep inside run_pipeline.
    _missing = _check_dataset_paths(_ds_cfg)
    if _missing:
        print(f"\n[{_ds_name}] SKIPPED -- path(s) not found: {_missing}")
        print(f"[{_ds_name}] Update DATASET_CONFIGS['{_ds_name}'] in the dataset-registry cell "
              f"above (or set DATA_ROOT) to point at your actual data location.\n")
        all_dataset_results[_ds_name] = {"error": f"path(s) not found: {_missing}"}
        continue

    # Pass the whole config dict through -- run_pipeline checks whether it has
    # "path" (flat) or "train_val_path"/"test_path" (pre-split) and loads accordingly.
    try:
        all_dataset_results[_ds_name] = run_pipeline(_ds_name, _ds_cfg)
    except Exception as e:
        print(f"\n[{_ds_name}] PIPELINE FAILED -- {type(e).__name__}: {e}")
        print(f"[{_ds_name}] Skipping to the next dataset (if any).\n")
        all_dataset_results[_ds_name] = {"error": str(e)}
    finally:
        torch.cuda.empty_cache()
        gc.collect()

print(f"\n\nDone. Ran: {list(all_dataset_results.keys())}")


In [ ]:
# Quick look at the headline numbers across every dataset that was run
for _ds_name, _res in all_dataset_results.items():
    print(f"\n=== {_ds_name} ===")
    if "error" in _res:
        print(f"  FAILED: {_res['error']}")
        continue
    print(_res["comparison_results"])
